# AI2002 - Group 5: Review-Based Hotel Question Answering Chatbot

> **Đề tài:** Joint Fine-Grained Opinion Extraction and Overall Rating Prediction from Real-World Hotel Reviews for Review-Based Question Answering Systems
>
> **Định vị trọng tâm:** Hệ thống Chatbot **Hỏi Đáp Chi Tiết Về Khách Sạn Dựa Trên Đánh Giá Thực Tế (Review-Based Hotel QA)**  *Tuyệt đối KHÔNG PHẢI hệ thống gợi ý / đề xuất khách sạn (Non-Recommendation).*


## 0. Cài đặt môi trường (Environment Setup & Auto-install)

Với dự án nghiên cứu về **Review Chatbot for Hotel Question Answering**, hệ thống được thiết lập cơ chế **tự động kiểm tra và cài đặt thư viện** từ file `requirements.txt` vào môi trường ảo (`.venv`):

1. **Tự động nhận diện môi trường ảo**: Cài đặt trực tiếp vào Python Kernel (`sys.executable`) đang chạy trong notebook.
2. **Kiểm tra thông minh**: Chỉ cài đặt các gói còn thiếu từ `requirements.txt`, tránh tốn thời gian khi chạy lại notebook.
3. **Các thư viện chính**:
   - **Pandas / NumPy**: Xử lý dữ liệu bảng quy mô lớn và tính toán đại số ma trận.
   - **PyArrow**: Engine C++ tối ưu hóa nạp dữ liệu siêu tốc và xử lý định dạng Parquet.
   - **SQLite3**: Cơ sở dữ liệu quan hệ cục bộ lưu trữ dữ liệu có cấu trúc.
   - **Matplotlib / Seaborn**: Trực quan hóa dữ liệu EDA và biểu đồ phân tích.
   - **Scikit-learn / NLTK**: Xử lý ngôn ngữ tự nhiên (NLP), TF-IDF và thuật toán Cosine Similarity.
   - **Tqdm / Ipywidgets**: Thanh tiến trình hiển thị trực quan trong quá trình xử lý văn bản.



In [1]:
import os
import re
import sqlite3
import pandas as pd
import numpy as np
import pyarrow as pa
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# Tự động tải các gói tài nguyên cần thiết cho NLTK 

# Hiển thị bảng phiên bản chi tiết các thư viện
print(f"{'Thư viện':<20} {'Phiên bản':>15}")
print("-" * 37)
print(f"{'Python':<20} {os.sys.version.split()[0]:>15}")
print(f"{'Pandas':<20} {pd.__version__:>15}")
print(f"{'PyArrow':<20} {pa.__version__:>15}")
print(f"{'NumPy':<20} {np.__version__:>15}")
print(f"{'Matplotlib':<20} {matplotlib.__version__:>15}")
print(f"{'Seaborn':<20} {sns.__version__:>15}")
print(f"{'Scikit-learn':<20} {sklearn.__version__:>15}")
print(f"{'SQLite3':<20} {sqlite3.sqlite_version:>15}")

# Khởi tạo cấu trúc thư mục lưu trữ dữ liệu
DATA_DIR = 'Data'
RAW_DIR = os.path.join(DATA_DIR, 'dts_raw')
DB_DIR = os.path.join(DATA_DIR, 'db')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
CHARTS_DIR = os.path.join(DATA_DIR, 'charts_img')

for directory in [DATA_DIR, RAW_DIR, DB_DIR, TRAIN_DIR, CHARTS_DIR]:
    os.makedirs(directory, exist_ok=True)

# Kết nối SQLite Database (ưu tiên Data/db/chatbot.db, tương thích ngược Data/chatbot.db)
db_path = os.path.join(DB_DIR, 'chatbot.db') if os.path.exists(os.path.join(DB_DIR, 'chatbot.db')) or not os.path.exists(os.path.join(DATA_DIR, 'chatbot.db')) else os.path.join(DATA_DIR, 'chatbot.db')
conn = sqlite3.connect(db_path)

print(f"\nSQLite đã sẵn sàng tại: {db_path}")
print("--- Môi trường đã được khởi tạo thành công ---")


# --- Imports thêm ---
import json
import random
from collections import defaultdict, Counter
from typing import Dict, List, Any, Tuple
import math
from typing import Dict, List, Tuple, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoConfig
from typing import List, Dict, Tuple, Any, Set
import time
from typing import List, Dict, Any, Tuple, Optional
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
import unicodedata
from typing import Dict, List, Any, Optional, Tuple
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from scipy.stats import pearsonr
from typing import List, Dict, Optional, Union
import hashlib
from IPython.display import display
from datetime import datetime
from collections import Counter
from tqdm import tqdm

Thư viện                   Phiên bản
-------------------------------------
Python                       3.12.14
Pandas                         3.0.6
PyArrow                       25.0.1
NumPy                         1.26.4
Matplotlib                    3.11.2
Seaborn                       0.13.2
Scikit-learn                   1.9.1
SQLite3                       3.53.4

SQLite đã sẵn sàng tại: Data/db/chatbot.db
--- Môi trường đã được khởi tạo thành công ---


## 1. Liên kết Dataset


### 1.1 CHỈ CHẠY TRÊN GOOGLE COLAB - không chạy trên VSCode (IDE) để tránh lỗi


### 1.2 Nhập và Kiểm tra Dataset (Dataset Loading & Exploration)

Trong phần này, nhóm áp dụng giải pháp **Tối ưu hóa nạp dữ liệu quy mô lớn** (~740MB, >780.000 dòng):
1. **Cơ chế Caching thông minh (Parquet Optimization)**:
   - Hệ thống tự động kiểm tra định dạng nhị phân dạng cột **Parquet** (`tripadvisor_review_hotel_dataset.parquet`). Nếu chưa tồn tại, chương trình sẽ tự động đọc CSV với engine **PyArrow** đa luồng và tạo cache Parquet (chuẩn nén Snappy).
2. **Kiểm tra cấu trúc và các trường thông tin quan trọng**:
   - **Thông tin cơ sở lưu trú**: `hotel_name`, `hotel_province`, `hotel_address`, `hotel_star`.
   - **Điểm đánh giá và các khía cạnh dịch vụ**: `normalized_score`, `Value`, `Rooms`, `Location`, `Cleanliness`, `Service`, `Sleep_Quality`.
   - **Dữ liệu văn bản phục vụ Chatbot Hỏi Đáp (QA)**: `normalized_title` (tiêu đề đánh giá), `normalized_content` (nội dung đánh giá chi tiết), `Word_count`, `language_code`, `language`.
   - **Bối cảnh chuyến đi & Thời gian**: `trip_type`, `Date`, `month`, `year`.



In [2]:
# Tạo thư mục lưu trữ biểu đồ trực quan hóa nếu chưa tồn tại
os.makedirs('Data/charts_img', exist_ok=True)

# Đường dẫn các file dữ liệu (ưu tiên cấu trúc chuẩn Data/dts_raw, tương thích ngược Data/)
parquet_path = (
    'Data/dts_raw/tripadvisor_review_hotel_dataset.parquet'
    if os.path.exists('Data/dts_raw/tripadvisor_review_hotel_dataset.parquet')
    else 'Data/tripadvisor_review_hotel_dataset.parquet'
)
csv_path = (
    'Data/dts_raw/tripadvisor_review_hotel_dataset.csv'
    if os.path.exists('Data/dts_raw/tripadvisor_review_hotel_dataset.csv')
    else 'Data/tripadvisor_review_hotel_dataset.csv'
)

start_time = time.time()

# Chiến lược nạp dữ liệu tối ưu với Parquet Cache
if os.path.exists(parquet_path):
    print(f"Tìm thấy cache Parquet: {parquet_path}")
    print("Đang nạp dữ liệu từ file Parquet...")
    df = pd.read_parquet(parquet_path)
    load_time = time.time() - start_time
    print(f"Nạp dữ liệu hoàn tất trong: {load_time:.2f} giây")
elif os.path.exists(csv_path):
    print(f"Tìm thấy file dataset CSV: {csv_path}")
    print("Đang đọc CSV và tạo cache Parquet...")
    df = pd.read_csv(csv_path, engine='pyarrow', encoding='utf-8')
    read_time = time.time() - start_time
    print(f"Đọc CSV thành công trong {read_time:.2f} giây")
    
    print("Đang lưu cache Parquet...")
    os.makedirs(os.path.dirname(parquet_path), exist_ok=True)
    df.to_parquet(parquet_path, engine='pyarrow', index=False)
    total_time = time.time() - start_time
    print(f"Đã tạo cache Parquet thành công tại: {parquet_path} (Thời gian: {total_time:.2f}s)")
else:
    raise FileNotFoundError(f"Không tìm thấy dataset tại '{csv_path}' hoặc '{parquet_path}'.")

# Thống kê tập dữ liệu
ram_usage_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
active_path = parquet_path if os.path.exists(parquet_path) else csv_path
file_size_mb = os.path.getsize(active_path) / (1024 ** 2)

print(f"\nKích thước tập dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")
print(f"Dung lượng file trên ổ cứng: {file_size_mb:.2f} MB")
print(f"Dung lượng RAM chiếm: {ram_usage_mb:.2f} MB")
print("\nCác cột dữ liệu:")
print(list(df.columns))

# Hiển thị 5 dòng đầu tiên
print("\nXem trước 5 dòng đầu tiên:")
display(df.head())


Tìm thấy cache Parquet: Data/dts_raw/tripadvisor_review_hotel_dataset.parquet
Đang nạp dữ liệu từ file Parquet...
Nạp dữ liệu hoàn tất trong: 1.94 giây

Kích thước tập dữ liệu: 782,584 dòng x 25 cột
Dung lượng file trên ổ cứng: 252.52 MB
Dung lượng RAM chiếm: 824.72 MB

Các cột dữ liệu:
['id_url', 'Date', 'month', 'year', 'month_str', 'normalized_score', 'trip_type', 'hotel_province', 'Value', 'Rooms', 'Location', 'Cleanliness', 'Service', 'Sleep_Quality', 'normalized_content', 'normalized_title', 'Word_count', 'language_code', 'language', 'nationality', 'hotel_name', 'hotel_address', 'source', 'hotel_star', 'link']

Xem trước 5 dòng đầu tiên:


,id_url,Date,month,year,month_str,normalized_score,trip_type,hotel_province,Value,Rooms,...,normalized_title,Word_count,language_code,language,nationality,hotel_name,hotel_address,source,hotel_star,link
0,https://www.tripadvisor.com.vn/Hotel_Review-g2...,2023-07,7,2023,Jul,5.0,Traveled as a couple,Hà Nội,5.0,NaN,...,Very good hotel,44,en,English,no_info,20 hotel and apartment,"93A Đội Cấn, Ba Dinh, Hà Nội 100000 Việt Nam",tripadvisor,3-star,https://www.tripadvisor.com.vn/Hotel_Review-g2...
1,https://www.tripadvisor.com.vn/Hotel_Review-g2...,2023-04,4,2023,Apr,4.0,Traveled as a couple,Hà Nội,4.0,NaN,...,BUEN ALOJAMIENTO QUE GANARIA MUCHO MEJORANDO E...,431,es,Spanish,Spain,20 hotel and apartment,"93A Đội Cấn, Ba Dinh, Hà Nội 100000 Việt Nam",tripadvisor,3-star,https://www.tripadvisor.com.vn/Hotel_Review-g2...
2,https://www.tripadvisor.com.vn/Hotel_Review-g2...,2023-05,5,2023,May,5.0,Traveled on business,Hà Nội,5.0,NaN,...,Great place in Cau Giay,71,en,English,Vietnam,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",tripadvisor,3-star,https://www.tripadvisor.com.vn/Hotel_Review-g2...
3,https://www.tripadvisor.com.vn/Hotel_Review-g2...,2023-04,4,2023,Apr,5.0,Traveled solo,Hà Nội,NaN,NaN,...,TRẢI NGHIỆM TỐT,45,vi,Vietnamese,Vietnam,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",tripadvisor,3-star,https://www.tripadvisor.com.vn/Hotel_Review-g2...
4,https://www.tripadvisor.com.vn/Hotel_Review-g2...,2022-12,12,2022,Dec,5.0,Traveled with family,Hà Nội,NaN,NaN,...,Perfect stay,44,en,English,Netherlands,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",tripadvisor,3-star,https://www.tripadvisor.com.vn/Hotel_Review-g2...


## 5. Pipeline Tiền Xử Lý Dữ Liệu Đa Ngôn Ngữ & Lọc Khách Sạn (Multilingual & Hotel Filtering Pipeline)

Phần này thực hiện quy trình tiền xử lý phục vụ **Huấn luyện Đa ngôn ngữ trực tiếp (Native Multilingual Training)** thay vì dịch thuật sang tiếng Anh, bảo tồn trọn vẹn ngữ nghĩa và ranh giới từ (Span boundaries) cho bài toán Aspect-Based Sentiment Analysis (ABSA/ASTE).

Đồng thời, tích hợp bộ công cụ lọc khách sạn đa tiêu chí:
1. **Lọc theo tên khách sạn** (`hotel_name`): Tìm kiếm chính xác hoặc theo từ khóa/tiền tố tên khách sạn.
2. **Lọc theo địa chỉ / khu vực** (`hotel_address`, `hotel_province`): Phân loại chính xác theo vị trí địa lý, chống nhầm lẫn giữa các cơ sở trùng tên.
3. **Lọc và phân tích ngôn ngữ review** (`language_code`, `language`): Bảo tồn dữ liệu đa ngôn ngữ hoặc lọc giữ các nhóm ngôn ngữ mục tiêu (tiếng Việt, tiếng Anh, v.v.).
4. **Các bước lọc chất lượng cao**: Loại bỏ review rỗng, lọc độ dài (101000 từ), loại trùng lặp nội dung, và lọc khách sạn ít review dựa trên cặp `(hotel_name, hotel_address)` để xuất file sạch `Data/train/reviews_filtered.jsonl`.


### 5.1. Module Lọc Khách Sạn & Phân Tích Đa Ngôn Ngữ (Hotel & Multilingual Filtering Engine)

Thay vì chuyển đổi toàn bộ ngôn ngữ về tiếng Anh bằng máy dịch (dễ gây nghẽn mạng, dính rate limit API và làm sai lệch vị trí span của từ khóa cảm xúc/khía cạnh), hệ thống áp dụng chiến lược **Native Multilingual Data Processing**.

Module này xây dựng class `HotelMultilingualFilter` cho phép:
- **Lọc theo tên khách sạn (`hotel_name`)**: Hỗ trợ lọc chính xác (exact match) hoặc tìm kiếm theo từ khóa/chuỗi con không phân biệt hoa thường.
- **Lọc theo địa chỉ (`hotel_address`, `hotel_province`)**: Lọc theo địa chỉ chi tiết hoặc theo tỉnh/thành phố.
- **Lọc theo ngôn ngữ review (`language_code`, `language`)**: Lọc một hoặc nhiều ngôn ngữ (ví dụ: `['vi', 'en']`, `['vi', 'en', 'ko', 'ja']` hoặc giữ tất cả).
- **Định danh khách sạn chuẩn xác**: Kết hợp cặp `(hotel_name, hotel_address)` để phân biệt các khách sạn cùng tên nhưng khác cơ sở/địa chỉ.
- **Thống kê ma trận phân bổ ngôn ngữ**: Phân tích tỷ lệ review theo từng ngôn ngữ cho từng khách sạn.


In [ ]:
# =====================================================================
# 5.1. Module Lọc Khách Sạn & Phân Tích Đa Ngôn Ngữ (Hotel & Multilingual Engine)
# =====================================================================


#  Đường dẫn dữ liệu 
DATA_DIR = os.path.join(os.getcwd(), "Data")
RAW_DIR = os.path.join(DATA_DIR, "dts_raw")
PARQUET_PATH = os.path.join(RAW_DIR, "tripadvisor_review_hotel_dataset.parquet")
CSV_PATH = os.path.join(RAW_DIR, "tripadvisor_review_hotel_dataset.csv")


class HotelMultilingualFilter:
    """
    Module lọc và phân tích dữ liệu khách sạn đa ngôn ngữ phục vụ Native Multilingual Training:
    - Lọc đánh giá theo Tên khách sạn (hotel_name)
    - Lọc đánh giá theo Địa chỉ / Tỉnh thành (hotel_address, hotel_province)
    - Lọc đánh giá theo Ngôn ngữ (language_code / language)
    - Định danh khách sạn bằng cặp (hotel_name, hotel_address) để chống trùng tên khác vị trí
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()

    def filter_by_hotel_name(
        self,
        names: Union[str, List[str]],
        exact_match: bool = False,
        case_sensitive: bool = False,
    ) -> pd.DataFrame:
        """
        Lọc đánh giá theo tên khách sạn.
        - exact_match: True nếu yêu cầu trùng khớp 100%, False nếu tìm kiếm từ khóa con.
        """
        if isinstance(names, str):
            names = [names]
        if not names:
            return self.df

        if exact_match:
            if not case_sensitive:
                names_lower = [n.strip().lower() for n in names]
                mask = (
                    self.df["hotel_name"]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .isin(names_lower)
                )
            else:
                names_set = set(n.strip() for n in names)
                mask = self.df["hotel_name"].astype(str).str.strip().isin(names_set)
        else:
            pattern = "|".join(re.escape(n.strip()) for n in names if n.strip())
            mask = (
                self.df["hotel_name"]
                .astype(str)
                .str.contains(pattern, case=case_sensitive, na=False)
            )

        return self.df[mask].copy()

    def filter_by_address(
        self,
        addresses: Optional[Union[str, List[str]]] = None,
        provinces: Optional[Union[str, List[str]]] = None,
        case_sensitive: bool = False,
    ) -> pd.DataFrame:
        """
        Lọc đánh giá theo địa chỉ chi tiết (hotel_address) hoặc tỉnh/thành phố (hotel_province).
        """
        res = self.df.copy()
        if addresses is not None:
            if isinstance(addresses, str):
                addresses = [addresses]
            if addresses:
                pattern = "|".join(re.escape(a.strip()) for a in addresses if a.strip())
                res = res[
                    res["hotel_address"]
                    .astype(str)
                    .str.contains(pattern, case=case_sensitive, na=False)
                ]

        if provinces is not None and "hotel_province" in res.columns:
            if isinstance(provinces, str):
                provinces = [provinces]
            if provinces:
                pattern_prov = "|".join(
                    re.escape(p.strip()) for p in provinces if p.strip()
                )
                res = res[
                    res["hotel_province"]
                    .astype(str)
                    .str.contains(pattern_prov, case=case_sensitive, na=False)
                ]

        return res

    def filter_by_language(self, languages: Union[str, List[str]]) -> pd.DataFrame:
        """
        Lọc đánh giá theo mã ngôn ngữ (language_code) hoặc tên ngôn ngữ (language).
        Ví dụ: languages=['en', 'vi', 'ko', 'fr']
        """
        if isinstance(languages, str):
            languages = [languages]
        if not languages:
            return self.df

        langs_lower = [l.strip().lower() for l in languages]
        mask_code = self.df["language_code"].astype(str).str.lower().isin(langs_lower)
        mask_name = (
            self.df["language"].astype(str).str.lower().isin(langs_lower)
            if "language" in self.df.columns
            else False
        )
        return self.df[mask_code | mask_name].copy()

    def filter_dataset(
        self,
        hotel_names: Optional[Union[str, List[str]]] = None,
        addresses: Optional[Union[str, List[str]]] = None,
        provinces: Optional[Union[str, List[str]]] = None,
        languages: Optional[Union[str, List[str]]] = None,
        min_hotel_reviews: int = 1,
        name_exact_match: bool = False,
    ) -> pd.DataFrame:
        """
        Bộ lọc tích hợp toàn diện: Tên khách sạn + Địa chỉ + Ngôn ngữ review + Ngưỡng review tối thiểu.
        """
        filtered = self.df.copy()

        # 1. Lọc theo ngôn ngữ
        if languages is not None:
            sub_filter = HotelMultilingualFilter(filtered)
            filtered = sub_filter.filter_by_language(languages)

        # 2. Lọc theo tên khách sạn
        if hotel_names is not None:
            sub_filter = HotelMultilingualFilter(filtered)
            filtered = sub_filter.filter_by_hotel_name(
                hotel_names, exact_match=name_exact_match
            )

        # 3. Lọc theo địa chỉ / tỉnh thành
        if addresses is not None or provinces is not None:
            sub_filter = HotelMultilingualFilter(filtered)
            filtered = sub_filter.filter_by_address(
                addresses=addresses, provinces=provinces
            )

        # 4. Lọc số lượng review tối thiểu theo cặp (hotel_name, hotel_address)
        if min_hotel_reviews > 1:
            hotel_counts = filtered.groupby(["hotel_name", "hotel_address"]).size()
            valid_keys = hotel_counts[hotel_counts >= min_hotel_reviews].index
            multi_idx = filtered.set_index(["hotel_name", "hotel_address"])
            filtered = filtered[multi_idx.index.isin(valid_keys)].copy()

        return filtered

    def get_language_matrix(
        self, top_n_hotels: int = 10, top_n_langs: int = 8
    ) -> pd.DataFrame:
        """
        Thống kê phân bổ ngôn ngữ đánh giá cho các khách sạn có nhiều review nhất.
        """
        top_hotels = self.df.groupby("hotel_name").size().nlargest(top_n_hotels).index
        top_langs = (
            self.df["language_code"].value_counts().nlargest(top_n_langs).index
        )

        subset = self.df[
            self.df["hotel_name"].isin(top_hotels)
            & self.df["language_code"].isin(top_langs)
        ]
        matrix = pd.crosstab(
            subset["hotel_name"],
            subset["language_code"],
            margins=True,
            margins_name="Total",
        )
        return matrix

    def print_multilingual_summary(self) -> None:
        """
        In bảng tổng quan dữ liệu đa ngôn ngữ và thống kê định danh khách sạn.
        """
        total_reviews = len(self.df)
        total_hotels = (
            self.df["hotel_name"].nunique() if "hotel_name" in self.df.columns else 0
        )
        total_pairs = (
            self.df.groupby(["hotel_name", "hotel_address"]).ngroups
            if "hotel_address" in self.df.columns
            else total_hotels
        )
        total_langs = (
            self.df["language_code"].nunique()
            if "language_code" in self.df.columns
            else 0
        )

        print(f"\n{'='*65}")
        print(f" TỔNG QUAN DỮ LIỆU ĐA NGÔN NGỮ & KHÁCH SẠN")
        print(f"{'='*65}")
        print(f"   Tổng số đánh giá (reviews):         {total_reviews:>10,}")
        print(f"   Số tên khách sạn độc lập:          {total_hotels:>10,}")
        print(f"   Số cơ sở định danh (Tên + Địa chỉ): {total_pairs:>10,}")
        print(f"   Số lượng ngôn ngữ review:          {total_langs:>10,}")

        if "language_code" in self.df.columns:
            print(f"\n   Top 10 ngôn ngữ chiếm tỷ trọng cao nhất:")
            lang_counts = self.df["language_code"].value_counts().head(10)
            for code, cnt in lang_counts.items():
                pct = cnt / total_reviews * 100
                bar = "" * int(pct / 5) + "" * (20 - int(pct / 5))
                lang_name = (
                    self.df[self.df["language_code"] == code]["language"].iloc[0]
                    if "language" in self.df.columns
                    else code
                )
                print(
                    f"      {code:6s} ({lang_name:<12s}): {bar} {cnt:>8,} ({pct:>5.1f}%)"
                )
        print(f"{'='*65}\n")



 TỔNG QUAN DỮ LIỆU ĐA NGÔN NGỮ & KHÁCH SẠN
   Tổng số đánh giá (reviews):            782,584
   Số tên khách sạn độc lập:              12,192
   Số cơ sở định danh (Tên + Địa chỉ):     12,656
   Số lượng ngôn ngữ review:                  43

   Top 10 ngôn ngữ chiếm tỷ trọng cao nhất:
      en     (English     ): █████████████░░░░░░░  527,338 ( 67.4%)
      vi     (Vietnamese  ): █░░░░░░░░░░░░░░░░░░░   48,396 (  6.2%)
      ko     (Korean      ): █░░░░░░░░░░░░░░░░░░░   42,547 (  5.4%)
      fr     (French      ): ░░░░░░░░░░░░░░░░░░░░   38,770 (  5.0%)
      de     (German      ): ░░░░░░░░░░░░░░░░░░░░   25,736 (  3.3%)
      ru     (Russian     ): ░░░░░░░░░░░░░░░░░░░░   22,612 (  2.9%)
      ja     (Japanese    ): ░░░░░░░░░░░░░░░░░░░░   18,552 (  2.4%)
      es     (Spanish     ): ░░░░░░░░░░░░░░░░░░░░   14,517 (  1.9%)
      nl     (Dutch       ): ░░░░░░░░░░░░░░░░░░░░   11,360 (  1.5%)
      it     (Italian     ): ░░░░░░░░░░░░░░░░░░░░    9,020 (  1.2%)



### 5.2. Cấu hình hệ thống (Configuration)

File cấu hình tập trung chứa các đường dẫn, ngưỡng lọc chất lượng dữ liệu, thiết lập chiến lược đa ngôn ngữ `LANGUAGE_STRATEGY = 'multilingual'`, danh sách lọc khách sạn theo tên và địa chỉ, cùng danh sách khía cạnh (Aspects) và phân vùng địa lý.


In [40]:
# =====================================================================
# 5.2. Cấu hình hệ thống tập trung cho Data Pipeline
# =====================================================================


#  Đường dẫn dữ liệu 
DATA_DIR = os.path.join(os.getcwd(), "Data")

# Input (raw data - ưu tiên Parquet đa ngôn ngữ gốc)
RAW_DIR = os.path.join(DATA_DIR, "dts_raw")
PARQUET_PATH = os.path.join(RAW_DIR, "tripadvisor_review_hotel_dataset.parquet")
CSV_PATH = os.path.join(RAW_DIR, "tripadvisor_review_hotel_dataset.csv")

# Output
NEW_DB_PATH = os.path.join(DATA_DIR, "db", "chatbot.db")
TRAINING_DIR = os.path.join(DATA_DIR, "train")

#  Cấu hình Chiến lược Đa Ngôn Ngữ (Multilingual) 
# Chiến lược: "multilingual" (toàn bộ ngôn ngữ) | "en_vi" | "en_only" | "custom"
LANGUAGE_STRATEGY = "multilingual"

# Mapping chiến lược  danh sách language codes giữ lại (None = giữ 100% tất cả ngôn ngữ)
LANGUAGE_FILTERS = {
    "multilingual": None,             # Giữ nguyên bản toàn bộ ngôn ngữ phục vụ Native Multilingual Training
    "en_vi": ["en", "vi"],            # Tập trung 2 ngôn ngữ chính: Tiếng Anh & Tiếng Việt
    "en_vi_major": ["en", "vi", "ko", "fr", "ja", "zh-cn", "de", "es"], # Nhóm ngôn ngữ du lịch phổ biến
    "en_only": ["en"],                # Đơn ngữ tiếng Anh (tùy chọn so sánh baseline)
}

#  Cấu hình Lọc Khách sạn theo Tên & Địa chỉ 
# Lọc theo tên khách sạn (None = lấy tất cả; hoặc danh sách từ khóa/tên khách sạn cần giữ)
FILTER_HOTEL_NAMES = None       # Ví dụ: ["Vinpearl", "Muong Thanh", "InterContinental"] hoặc None

# Lọc theo địa chỉ chi tiết (None = lấy tất cả; hoặc danh sách chuỗi địa chỉ cần giữ)
FILTER_HOTEL_ADDRESSES = None   # Ví dụ: ["Đà Nẵng", "Hà Nội", "Nha Trang"] hoặc None

# Lọc theo tỉnh thành (None = lấy tất cả; hoặc danh sách tỉnh thành)
FILTER_HOTEL_PROVINCES = None   # Ví dụ: ["Đà Nẵng", "Khánh Hòa", "Hà Nội"] hoặc None

#  Lọc chất lượng review & định danh khách sạn 
MIN_WORD_COUNT = 10             # Loại review quá ngắn (ví dụ "Good", "Nice hotel")
MAX_WORD_COUNT = 1000           # Loại review spam/copy-paste cực dài
REMOVE_DUPLICATES = True        # Loại review trùng nội dung
MIN_HOTEL_REVIEWS = 5           # Loại KS có quá ít review (định danh theo cặp Name + Address)
REMOVE_NULL_CONTENT = True       # Loại review không có nội dung

#  Aspect scores 
ASPECT_COLUMNS = [
    "Value", "Rooms", "Location",
    "Cleanliness", "Service", "Sleep_Quality",
]

# Mapping tên aspect  mô tả tiếng Việt (cho chatbot response)
ASPECT_LABELS_VI = {
    "Value": "Giá trị",
    "Rooms": "Phòng ốc",
    "Location": "Vị trí",
    "Cleanliness": "Vệ sinh",
    "Service": "Dịch vụ",
    "Sleep_Quality": "Chất lượng giấc ngủ",
}

#  Phân vùng địa lý 
REGIONS = {
    "Hà Nội": ["Hà Nội"],
    "TP.HCM": ["Thành phố Hồ Chí Minh"],
    "Miền Trung": ["Quảng Nam", "Đà Nẵng", "Thừa Thiên Huế", "Khánh Hòa"],
    "Phú Quốc & Mekong": ["Kiên Giang", "Cần Thơ"],
    "Biển & Resort": ["Bình Thuận", "Bà Rịa - Vũng Tàu"],
    "Tây Nguyên": ["Lâm Đồng"],
    "Miền Bắc khác": ["Ninh Bình", "Hải Phòng", "Quảng Ninh", "Quảng Bình"],
}

# Reverse mapping: province  region (Chuẩn hóa Unicode NFC để tránh lệch NFD)

PROVINCE_TO_REGION = {}
for region, provinces in REGIONS.items():
    for province in provinces:
        PROVINCE_TO_REGION[unicodedata.normalize("NFC", province)] = region

#  Cân bằng dữ liệu (cho training) 
# Phương pháp: "class_weights" | "undersample" | "oversample" | "none"
BALANCE_METHOD = "class_weights"

#  Cấu hình text preprocessing 
# Cho Classical ML (TF-IDF, BM25):
TEXT_PREPROCESSING_CLASSICAL = {
    "lowercase": True,
    "remove_html": True,
    "remove_special_chars": True,
    "remove_stopwords": True,
    "lemmatize": True,
}

# Cho Multilingual Transformer models (mDeBERTa, XLM-RoBERTa):
TEXT_PREPROCESSING_TRANSFORMER = {
    "lowercase": False,             # Tokenizer đa ngôn ngữ tự xử lý cased/uncased
    "remove_html": True,
    "remove_special_chars": False,  # Giữ nguyên context và punctuation cho các ngôn ngữ
    "remove_stopwords": False,      # Transformer cần full context liên từ/sắc thái
    "lemmatize": False,             # Tokenizer Subword/BPE đa ngôn ngữ
}

#  Positive / Negative keywords 
POSITIVE_KEYWORDS = [
    "excellent", "amazing", "wonderful", "fantastic", "great", "perfect",
    "beautiful", "lovely", "outstanding", "superb", "exceptional", "brilliant",
    "comfortable", "friendly", "helpful", "clean", "spacious", "delicious",
    "recommend", "love", "best", "enjoy", "pleasant", "impressive",
    "tuyệt vời", "xuất sắc", "rất tốt", "sạch sẽ", "nhiệt tình", "hài lòng",
]

NEGATIVE_KEYWORDS = [
    "terrible", "horrible", "awful", "worst", "disgusting", "dirty",
    "noisy", "rude", "disappointing", "uncomfortable", "broken", "poor",
    "bad", "smell", "cockroach", "bug", "mold", "stain", "overpriced",
    "never", "avoid", "complaint", "unacceptable", "nightmare",
    "tệ", "kinh khủng", "thất vọng", "bẩn", "ồn ào", "thô lỗ", "kém",
]

#  Hotel ID generation 
HOTEL_ID_SEED = 42


# =========================================================================
# JUPYTER COMPATIBILITY: Tạo object `cfg` để tương thích với Pipeline
# =========================================================================
class ConfigObj:
    pass

cfg = ConfigObj()
for key, value in dict(globals()).items():
    if key.isupper():
        setattr(cfg, key, value)

print(f'[OK] Đã nạp cấu hình Data Pipeline Đa Ngôn Ngữ thành công:')
print(f'    Chiến lược ngôn ngữ: {cfg.LANGUAGE_STRATEGY}')
print(f'    Lọc theo tên khách sạn: {cfg.FILTER_HOTEL_NAMES if cfg.FILTER_HOTEL_NAMES else "Tất cả"}')
print(f'    Lọc theo địa chỉ: {cfg.FILTER_HOTEL_ADDRESSES if cfg.FILTER_HOTEL_ADDRESSES else "Tất cả"}')
print(f'    Ngưỡng lọc độ dài từ: {cfg.MIN_WORD_COUNT} - {cfg.MAX_WORD_COUNT}')
print(f'    Số review tối thiểu/khách sạn (Tên + Đ/c): {cfg.MIN_HOTEL_REVIEWS}')
print(f'    Thư mục xuất dữ liệu: {cfg.TRAINING_DIR}')


[OK] Đã nạp cấu hình Data Pipeline Đa Ngôn Ngữ thành công:
   ├─ Chiến lược ngôn ngữ: multilingual
   ├─ Lọc theo tên khách sạn: Tất cả
   ├─ Lọc theo địa chỉ: Tất cả
   ├─ Ngưỡng lọc độ dài từ: 10 - 1000
   ├─ Số review tối thiểu/khách sạn (Tên + Đ/c): 5
   └─ Thư mục xuất dữ liệu: /Users/trietnguyen/Documents/Hotel-Review-AI-Chatbot/Data/train


### 5.3. Pipeline Tiền Xử Lý & Chuẩn Bị Dữ Liệu Sạch Đa Ngôn Ngữ (Data Pipeline)

Thực hiện quy trình 6 bước lọc dữ liệu chất lượng cao và chuẩn hóa định danh phục vụ huấn luyện đa ngôn ngữ:
1. **Lọc ngôn ngữ review**: Giữ lại dữ liệu theo `LANGUAGE_STRATEGY` (mặc định `'multilingual'` bảo tồn trọn vẹn mọi ngôn ngữ nguyên bản).
2. **Lọc theo tên khách sạn & địa chỉ**:
   - Loại bỏ các dòng thiếu tên khách sạn hoặc địa chỉ.
   - Áp dụng bộ lọc theo tên (`FILTER_HOTEL_NAMES`) và địa chỉ/tỉnh thành (`FILTER_HOTEL_ADDRESSES`, `FILTER_HOTEL_PROVINCES`) nếu được cấu hình.
3. **Loại review null/empty**: Loại bỏ các review không có nội dung văn bản (`normalized_content`).
4. **Lọc theo độ dài từ**: Giữ review có độ dài 10 đến 1000 từ (`10 <= Word_count <= 1000`).
5. **Loại bỏ trùng lặp**: Loại bỏ review trùng lặp nội dung (`normalized_content`).
6. **Lọc khách sạn ít review**: Định danh chính xác từng cơ sở khách sạn bằng cặp `(hotel_name, hotel_address)` và loại bỏ các khách sạn có ít hơn `MIN_HOTEL_REVIEWS` bài đánh giá.
7. **Chuẩn hóa định danh & Đường link URL**:
   - Gán mã định danh duy nhất khách sạn `hotel_id` = `hash(hotel_name + hotel_address)`.
   - Gán mã định danh duy nhất bài đánh giá `review_id` (1 .. N).
   - Ánh xạ phân vùng địa lý `hotel_region` từ `hotel_province`.
   - Trích xuất mã định danh khách sạn TripAdvisor `tripadvisor_hotel_id` từ URL `-d(\d+)-`.
   - Chuẩn hóa đường link `link` và `id_url` hiển thị chuẩn `https://...` (loại bỏ hoàn toàn lỗi escape gạch chéo `\/` của Pandas `to_json`).

Sau khi lọc, pipeline xuất trực tiếp file JSON Lines tiêu chuẩn `Data/train/reviews_filtered.jsonl`  tối ưu cho các mô hình học sâu NLP (PyTorch, Hugging Face) và khâu trích xuất đặc trưng.


In [41]:
# =====================================================================
# 5.3. Pipeline Tiền Xử Lý & Lọc Dữ Liệu Đa Ngôn Ngữ Chất Lượng Cao
# =====================================================================



# 
# 1. LOAD DATA
# 


def load_data() -> pd.DataFrame:
    """
    Nạp dữ liệu từ Parquet hoặc CSV gốc (nguyên bản đa ngôn ngữ).
    """
    if os.path.exists(cfg.PARQUET_PATH):
        print(f" Loading Parquet: {cfg.PARQUET_PATH}")
        t0 = time.time()
        df = pd.read_parquet(cfg.PARQUET_PATH)
        print(f"  [OK] Loaded {len(df):,} rows in {time.time() - t0:.1f}s")
    elif os.path.exists(cfg.CSV_PATH):
        print(f" Loading CSV: {cfg.CSV_PATH}")
        t0 = time.time()
        df = pd.read_csv(cfg.CSV_PATH)
        print(f"  [OK] Loaded {len(df):,} rows in {time.time() - t0:.1f}s")
    else:
        raise FileNotFoundError(
            f"Không tìm thấy dataset tại {cfg.PARQUET_PATH} hoặc {cfg.CSV_PATH}"
        )
    return df


# 
# 2. GENERATE HOTEL IDs & EXPORT JSONL CHUẨN
# 


def generate_hotel_id(name: str, address: str) -> int:
    """
    Tạo hotel_id duy nhất từ hash(name + address) chuẩn hóa Unicode NFC.
    Tránh trùng tên KS ở các tỉnh hoặc vị trí khác nhau.
    """
    name_norm = unicodedata.normalize("NFC", str(name).strip().lower())
    addr_norm = unicodedata.normalize("NFC", str(address).strip().lower())
    key = f"{name_norm}|{addr_norm}"
    return int(hashlib.sha256(key.encode("utf-8")).hexdigest()[:15], 16)


def export_df_to_jsonl(
    df: pd.DataFrame, file_path: str, chunk_size: int = 50000
) -> None:
    """
    Xuất DataFrame ra file JSON Lines (.jsonl) chuẩn:
    - Loại bỏ hoàn toàn lỗi escape ký tự gạch chéo (/ -> \/) của Pandas to_json.
    - Giữ đường link đúng định dạng chuẩn: https://...
    - Tối ưu bộ nhớ RAM bằng cách ghi theo chunk.
    """
    os.makedirs(os.path.dirname(os.path.abspath(file_path)), exist_ok=True)
    total_rows = len(df)
    with open(file_path, "w", encoding="utf-8") as f:
        for start in range(0, total_rows, chunk_size):
            chunk = df.iloc[start : start + chunk_size]
            chunk_json = chunk.to_json(orient="records", lines=True, force_ascii=False)
            clean_chunk = chunk_json.replace(r"\/", "/")
            f.write(clean_chunk)


# 
# 3. FILTER REVIEWS (Lọc đa ngôn ngữ, chất lượng & bổ sung định danh)
# 


def filter_reviews(df: pd.DataFrame) -> pd.DataFrame:
    """
    Áp dụng 6 bước lọc chất lượng và chuẩn hóa định danh:
    1. Ngôn ngữ review (bảo tồn đa ngôn ngữ)
    2. Tên khách sạn & Địa chỉ (chính xác và loại bỏ rỗng)
    3. Nội dung review rỗng
    4. Độ dài từ (10-1000 từ)
    5. Trùng lặp nội dung
    6. Số lượng review tối thiểu theo cặp (hotel_name, hotel_address)
    7. Bổ sung các trường định danh: hotel_id, review_id, hotel_region, tripadvisor_hotel_id
    """
    original_count = len(df)
    print(f"\n{'='*60}")
    print(f" BẮT ĐẦU LỌC DỮ LIỆU ĐA NGÔN NGỮ ({original_count:,} reviews)")
    print(f"{'='*60}")

    #  3.1. Lọc ngôn ngữ review 
    lang_codes = cfg.LANGUAGE_FILTERS.get(cfg.LANGUAGE_STRATEGY)
    if lang_codes is not None:
        before = len(df)
        df = df[df["language_code"].isin(lang_codes)].copy()
        removed = before - len(df)
        print(f"\n [1/6] Lọc ngôn ngữ review (chiến lược: {cfg.LANGUAGE_STRATEGY})")
        print(f"   Giữ ngôn ngữ: {lang_codes}")
        print(f"   Loại: {removed:,} reviews | Còn lại: {len(df):,}")
    else:
        n_langs = df["language_code"].nunique() if "language_code" in df.columns else 0
        print(f"\n [1/6] Lọc ngôn ngữ review: BẢO TỒN TOÀN BỘ ĐA NGÔN NGỮ (Multilingual)")
        print(f"   Số ngôn ngữ ghi nhận: {n_langs} mã ngôn ngữ")

    #  3.2. Lọc theo tên khách sạn & địa chỉ 
    before = len(df)
    df = df[
        df["hotel_name"].notna() & (df["hotel_name"].astype(str).str.strip() != "")
    ].copy()
    if "hotel_address" in df.columns:
        df = df[
            df["hotel_address"].notna()
            & (df["hotel_address"].astype(str).str.strip() != "")
        ].copy()

    # Lọc theo danh sách tên khách sạn nếu cấu hình
    if getattr(cfg, "FILTER_HOTEL_NAMES", None):
        target_names = [n.strip().lower() for n in cfg.FILTER_HOTEL_NAMES if n.strip()]
        pattern = "|".join(re.escape(n) for n in target_names)
        df = df[
            df["hotel_name"].astype(str).str.lower().str.contains(pattern, na=False)
        ].copy()
        print(f"   Áp dụng lọc theo tên khách sạn: {cfg.FILTER_HOTEL_NAMES}")

    # Lọc theo địa chỉ nếu cấu hình
    if getattr(cfg, "FILTER_HOTEL_ADDRESSES", None):
        target_addrs = [
            a.strip().lower() for a in cfg.FILTER_HOTEL_ADDRESSES if a.strip()
        ]
        pattern = "|".join(re.escape(a) for a in target_addrs)
        df = df[
            df["hotel_address"].astype(str).str.lower().str.contains(pattern, na=False)
        ].copy()
        print(f"   Áp dụng lọc theo địa chỉ khách sạn: {cfg.FILTER_HOTEL_ADDRESSES}")

    # Lọc theo tỉnh thành nếu cấu hình
    if getattr(cfg, "FILTER_HOTEL_PROVINCES", None) and "hotel_province" in df.columns:
        df = df[df["hotel_province"].isin(cfg.FILTER_HOTEL_PROVINCES)].copy()
        print(f"   Áp dụng lọc theo tỉnh thành: {cfg.FILTER_HOTEL_PROVINCES}")

    removed = before - len(df)
    print(f"\n [2/6] Lọc theo Tên khách sạn & Địa chỉ")
    print(f"   Loại: {removed:,} reviews | Còn lại: {len(df):,}")

    #  3.3. Loại review null content 
    if cfg.REMOVE_NULL_CONTENT:
        before = len(df)
        df = df[
            df["normalized_content"].notna()
            & (df["normalized_content"].astype(str).str.strip() != "")
        ].copy()
        removed = before - len(df)
        print(f"\n [3/6] Loại review null/empty content")
        print(f"   Loại: {removed:,} reviews | Còn lại: {len(df):,}")

    #  3.4. Lọc theo độ dài từ 
    before = len(df)
    df = df[
        (df["Word_count"] >= cfg.MIN_WORD_COUNT)
        & (df["Word_count"] <= cfg.MAX_WORD_COUNT)
    ].copy()
    removed = before - len(df)
    print(
        f"\n [4/6] Lọc độ dài ({cfg.MIN_WORD_COUNT}  word_count  {cfg.MAX_WORD_COUNT})"
    )
    print(f"   Loại: {removed:,} reviews | Còn lại: {len(df):,}")

    #  3.5. Loại review trùng lặp nội dung 
    if cfg.REMOVE_DUPLICATES:
        before = len(df)
        df = df.drop_duplicates(subset=["normalized_content"], keep="first").copy()
        removed = before - len(df)
        print(f"\n [5/6] Loại review trùng nội dung")
        print(f"   Loại: {removed:,} reviews | Còn lại: {len(df):,}")

    #  3.6. Lọc KS có quá ít review (Định danh theo Cặp Tên + Địa chỉ) 
    before = len(df)
    hotel_keys = df.groupby(["hotel_name", "hotel_address"]).size()
    valid_hotels = hotel_keys[hotel_keys >= cfg.MIN_HOTEL_REVIEWS].index

    # Giữ lại các reviews thuộc cặp (hotel_name, hotel_address) hợp lệ
    multi_idx = df.set_index(["hotel_name", "hotel_address"])
    df = df[multi_idx.index.isin(valid_hotels)].copy()

    removed = before - len(df)
    hotels_removed = len(hotel_keys) - len(valid_hotels)
    print(
        f"\n [6/6] Lọc KS có ít hơn {cfg.MIN_HOTEL_REVIEWS} reviews (định danh chuẩn Tên + Địa chỉ)"
    )
    print(
        f"   Loại: {removed:,} reviews ({hotels_removed:,} KS) | Còn lại: {len(df):,}"
    )
    print(f"   Số khách sạn giữ lại: {len(valid_hotels):,} khách sạn")

    #  3.7. Chuẩn hóa đường link & Bổ sung trường định danh 
    print(f"\n [7/7] Chuẩn hóa đường link & Bổ sung trường định danh:")
    # 1. Gán hotel_id duy nhất
    hotel_id_map = {}
    unique_hotels = df[["hotel_name", "hotel_address"]].drop_duplicates()
    for name, addr in unique_hotels.itertuples(index=False):
        hotel_id_map[(name, addr)] = generate_hotel_id(name, addr)
    df["hotel_id"] = [
        hotel_id_map.get((n, a)) for n, a in zip(df["hotel_name"], df["hotel_address"])
    ]

    # 2. Gán review_id duy nhất
    df["review_id"] = range(1, len(df) + 1)

    # 3. Gán hotel_region (chuẩn hóa NFC)
    if "hotel_province" in df.columns:
        norm_provinces = df["hotel_province"].astype(str).apply(lambda x: unicodedata.normalize("NFC", x))
        df["hotel_region"] = norm_provinces.map(getattr(cfg, "PROVINCE_TO_REGION", {})).fillna("Khác")

    # 4. Trích xuất TripAdvisor hotel ID từ đường link
    if "link" in df.columns:
        df["tripadvisor_hotel_id"] = df["link"].astype(str).str.extract(r"-d(\d+)-")[0]
    elif "id_url" in df.columns:
        df["tripadvisor_hotel_id"] = (
            df["id_url"].astype(str).str.extract(r"-d(\d+)-")[0]
        )

    # 5. Sắp xếp thứ tự cột logic và trực quan
    priority_cols = [
        "review_id",
        "hotel_id",
        "tripadvisor_hotel_id",
        "hotel_name",
        "hotel_address",
        "hotel_province",
        "hotel_region",
        "hotel_star",
        "link",
        "id_url",
        "normalized_score",
        "normalized_title",
        "normalized_content",
        "Word_count",
        "language_code",
        "language",
        "trip_type",
        "Date",
        "year",
        "month",
        "month_str",
        "nationality",
        "source",
        "Value",
        "Rooms",
        "Location",
        "Cleanliness",
        "Service",
        "Sleep_Quality",
    ]
    ordered_cols = [c for c in priority_cols if c in df.columns] + [
        c for c in df.columns if c not in priority_cols
    ]
    df = df[ordered_cols].copy()
    print(f"   [OK] Đã gắn: review_id, hotel_id, hotel_region, tripadvisor_hotel_id")
    print(f"   [OK] Đã chuẩn hóa thứ tự các trường ({len(ordered_cols)} cột)")

    total_removed = original_count - len(df)
    retention_pct = len(df) / original_count * 100 if original_count > 0 else 0
    print(f"\n{''*60}")
    print(f"[STATS] TỔNG KẾT LỌC DỮ LIỆU ĐA NGÔN NGỮ:")
    print(f"   Gốc:      {original_count:>8,} reviews")
    print(f"   Đã loại:  {total_removed:>8,} reviews ({100 - retention_pct:.1f}%)")
    print(f"   Giữ lại:  {len(df):>8,} reviews ({retention_pct:.1f}%)")
    print(f"{''*60}\n")

    return df


def run_preprocessing_pipeline() -> pd.DataFrame:
    """Chạy pipeline tiền xử lý đa ngôn ngữ và lưu reviews_filtered.jsonl chuẩn định dạng."""
    t_start = time.time()
    print(f"\n{''*60}")
    print(f" CHẠY PIPELINE TIỀN XỬ LÝ & LỌC DỮ LIỆU ĐA NGÔN NGỮ")
    print(f"{''*60}")
    os.makedirs(cfg.TRAINING_DIR, exist_ok=True)

    df_raw = load_data()
    df_filtered = filter_reviews(df_raw)

    jsonl_path = os.path.join(cfg.TRAINING_DIR, "reviews_filtered.jsonl")
    export_df_to_jsonl(df_filtered, jsonl_path)
    
    # Save to SQLite database for Section 6 and training
    import sqlite3
    db_conn = sqlite3.connect("Data/db/chatbot.db")
    df_filtered.to_sql("cleaned_hotel_reviews", db_conn, if_exists="replace", index=False)
    db_conn.close()
    
    size_mb = os.path.getsize(jsonl_path) / (1024 * 1024)
    print(
        f"\n[OK] Đã lưu dữ liệu sạch đa ngôn ngữ vào JSONL ({jsonl_path}) và CSDL (chatbot.db)"
    )
    print(f"   Số dòng: {len(df_filtered):,} ({size_mb:.1f} MB)")
    print(f"[TIME]  Thời gian xử lý: {time.time() - t_start:.1f}s")
    return df_filtered


<>:61: SyntaxWarning: invalid escape sequence '\/'
<>:61: SyntaxWarning: invalid escape sequence '\/'
/var/folders/m_/115k21gx4c12qm05_3nsm0hw0000gn/T/ipykernel_2021/3518990219.py:61: SyntaxWarning: invalid escape sequence '\/'
  - Loại bỏ hoàn toàn lỗi escape ký tự gạch chéo (/ -> \/) của Pandas to_json.


### 5.4. Chạy Pipeline Tiền Xử Lý & Lọc Dữ Liệu Đa Ngôn Ngữ (Thực thi)

Thực thi pipeline tiền xử lý và lọc dữ liệu đa ngôn ngữ để tạo ra tập dữ liệu sạch `Data/train/reviews_filtered.jsonl`.


In [89]:
# =====================================================================
# 5.4. Thực thi Pipeline Tiền Xử Lý & Lọc Dữ Liệu Đa Ngôn Ngữ
# =====================================================================

# 1. Chạy toàn bộ pipeline tiền xử lý và lọc
reviews_filtered_df = run_preprocessing_pipeline()

# 2. Hiển thị 5 dòng đầu tiên của tập dữ liệu sạch sau lọc
try:
    cols_preview = [c for c in ["hotel_name", "hotel_address", "language_code", "normalized_score", "normalized_content"] if c in reviews_filtered_df.columns]
    display(reviews_filtered_df[cols_preview].head(5))
except ImportError:
    cols_preview = [c for c in ["hotel_name", "hotel_address", "language_code", "normalized_score"] if c in reviews_filtered_df.columns]
    print(reviews_filtered_df[cols_preview].head(5))




 CHẠY PIPELINE TIỀN XỬ LÝ & LỌC DỮ LIỆU ĐA NGÔN NGỮ

 Loading Parquet: /Users/trietnguyen/Documents/Hotel-Review-AI-Chatbot/Data/dts_raw/tripadvisor_review_hotel_dataset.parquet
  [OK] Loaded 782,584 rows in 1.8s

 BẮT ĐẦU LỌC DỮ LIỆU ĐA NGÔN NGỮ (782,584 reviews)

 [1/6] Lọc ngôn ngữ review: BẢO TỒN TOÀN BỘ ĐA NGÔN NGỮ (Multilingual)
   Số ngôn ngữ ghi nhận: 43 mã ngôn ngữ

 [2/6] Lọc theo Tên khách sạn & Địa chỉ
   Loại: 4 reviews | Còn lại: 782,580

 [3/6] Loại review null/empty content
   Loại: 0 reviews | Còn lại: 782,580

 [4/6] Lọc độ dài (10  word_count  1000)
   Loại: 27,821 reviews | Còn lại: 754,759

 [5/6] Loại review trùng nội dung
   Loại: 48 reviews | Còn lại: 754,711

 [6/6] Lọc KS có ít hơn 5 reviews (định danh chuẩn Tên + Địa chỉ)
   Loại: 10,881 reviews (5,896 KS) | Còn lại: 743,830
   Số khách sạn giữ lại: 6,666 khách sạn

 [7/7] Chuẩn hóa đường link & Bổ sung trường định danh:
   [OK] Đã gắn: review_id, hotel_id, hotel_region, tripadvisor_hotel_id
   [OK] Đã chuẩ

,hotel_name,hotel_address,language_code,normalized_score,normalized_content
2,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",en,5.0,This place was very nice. Our bedroom were cle...
3,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",vi,5.0,Đầy đủ dịch vụ tiện nghi Ăn sáng buffee ngon H...
4,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",en,5.0,It was a amazing hotel. They helped very good ...
5,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",en,5.0,I was amazed at the hotel. The room was beauti...
8,22land residence hotel,"02 Nguyen Dinh Hoan, Cau Giay, Hà Nội 100000 V...",vi,5.0,"Khách sạn mới, sạch sẽ, có bar và bể bơi ở tần..."


## 6. Feature Engineering & Xây Dựng Cơ Sở Tri Thức Chatbot Hỏi Đáp (QA Knowledge Base)

Trong mục này, nhóm tiến hành xây dựng kho tri thức phục vụ trực tiếp cho hệ thống **Chatbot Hỏi Đáp Khách Sạn (Review-Based QA)**:
1. Trích xuất **10 đặc trưng (Features)** trên tập dữ liệu đã qua lọc sạch đa ngôn ngữ (`reviews_filtered.jsonl`), hỗ trợ đánh giá chất lượng, độ mới và xếp hạng mức độ hữu ích của review (Review Informativeness & Relevance Ranking) khi trích xuất dẫn chứng trả lời câu hỏi.
2. Xây dựng **Cơ sở dữ liệu hỏi đáp (QA Database)** trong SQLite (`chatbot.db`) gồm 3 bảng quan hệ (`hotels`, `reviews`, `hotel_summaries`) tổ chức tri thức theo từng cơ sở khách sạn và từng khía cạnh dịch vụ cụ thể.
3. Xuất tập dữ liệu huấn luyện Chatbot QA đa ngôn ngữ (`reviews_qa.jsonl`) làm Context Corpus phục vụ RAG và NLP fine-tuning cho bài toán Hỏi Đáp chi tiết.


Khởi tạo file features.jsonl tại `Data/train/features.jsonl` từ dữ liệu sạch đã lọc (`reviews_filtered.jsonl`)


In [90]:
train_dir = 'Data/train' if os.path.exists('Data/train') else 'Data'
os.makedirs(train_dir, exist_ok=True)
filtered_path = os.path.join(train_dir, 'reviews_filtered.jsonl')
features_path = os.path.join(train_dir, 'features.jsonl')

if not os.path.exists(filtered_path):
    print(f'[WARNING] Chưa tìm thấy {filtered_path}, đang chạy tiền xử lý...')
    df_raw = load_data()
    df_initial = filter_reviews(df_raw)
    export_df_to_jsonl(df_initial, filtered_path)
else:
    print(f' Đang nạp dữ liệu sạch JSONL từ: {filtered_path}')
    import sqlite3
    _local_conn = sqlite3.connect("Data/db/chatbot.db")
    df_initial = pd.read_sql_query('SELECT * FROM cleaned_hotel_reviews', _local_conn)
    _local_conn.close()

# Khởi tạo df để chạy Feature Engineering In-Memory
df = df_initial.copy()

# Khởi tạo features.jsonl đồng bộ 100% với dữ liệu sạch JSONL chuẩn
export_df_to_jsonl(df, features_path)

print(f'\n[OK] Tạo xong features.jsonl tại: {features_path}')
print(f'Số dòng: {df.shape[0]:,} | Số cột: {df.shape[1]}')
print('Cột:', list(df.columns))



 Đang nạp dữ liệu sạch JSONL từ: Data/train/reviews_filtered.jsonl

[OK] Tạo xong features.jsonl tại: Data/train/features.jsonl
Số dòng: 782,580 | Số cột: 31
Cột: ['id_url', 'Date', 'month', 'year', 'month_str', 'normalized_score', 'trip_type', 'hotel_province', 'Value', 'Rooms', 'Location', 'Cleanliness', 'Service', 'Sleep_Quality', 'normalized_content', 'normalized_title', 'Word_count', 'language_code', 'language', 'nationality', 'hotel_name', 'hotel_address', 'source', 'hotel_star', 'link', 'trip_type_Encoded', 'hotel_province_Encoded', 'language_Encoded', 'nationality_Encoded', 'source_Encoded', 'hotel_star_Encoded']


### 6.1. Feature 1: Điểm đánh giá trung bình theo Tỉnh & Loại chuyến đi (hotel_province_trip_type_Avg_Score)


Mục đích phục vụ Hỏi Đáp: Cung cấp thông tin bối cảnh phân khúc chuyến đi (công tác, gia đình, cặp đôi...) để Chatbot giải đáp chính xác khi người dùng hỏi các câu hỏi chuyên biệt như: *'Khách sạn này có phù hợp cho chuyến đi gia đình có trẻ nhỏ không?'* hoặc *'Khách đi công tác đánh giá khách sạn này ra sao?'*



In [91]:
# df already in memory

group_col_1 = "hotel_province"
group_col_2 = "trip_type"
value_col = "normalized_score"

feature_name = f"{group_col_1}_{group_col_2}_Avg_Score"
df[feature_name] = df.groupby([group_col_1, group_col_2])[value_col].transform('mean')

# df state is kept in memory
print(f"Feature 1: {feature_name}")
print(df[[group_col_1, group_col_2, feature_name]].head())


Feature 1: hotel_province_trip_type_Avg_Score
  hotel_province             trip_type  hotel_province_trip_type_Avg_Score
0       Hà Nội  Traveled as a couple                            4.667596
1       Hà Nội  Traveled as a couple                            4.667596
2       Hà Nội  Traveled on business                            3.977703
3       Hà Nội         Traveled solo                            4.022975
4       Hà Nội  Traveled with family                            4.128319


### 6.2. Feature 2: Điểm đánh giá trung bình theo Hạng sao khách sạn & Quốc tịch khách (hotel_star_nationality_Avg_Score)


Mục đích phục vụ Hỏi Đáp: Giúp Chatbot nắm bắt góc nhìn đa văn hóa và tiêu chuẩn đánh giá của từng tệp du khách khi người dùng hỏi về trải nghiệm theo quốc tịch (ví dụ: du khách quốc tế/châu Á đánh giá thế nào về khách sạn này).



In [92]:
# df already in memory

group_col_1 = "hotel_star"
group_col_2 = "nationality"
value_col = "normalized_score"

feature_name = f"{group_col_1}_{group_col_2}_Avg_Score"
df[feature_name] = df.groupby([group_col_1, group_col_2])[value_col].transform('mean')

# df state is kept in memory
print(f"Feature 2: {feature_name}")
print(df[[group_col_1, group_col_2, feature_name]].head())


Feature 2: hotel_star_nationality_Avg_Score
  hotel_star  nationality  hotel_star_nationality_Avg_Score
0     3-star      no_info                          4.626520
1     3-star        Spain                          4.341932
2     3-star      Vietnam                          4.427479
3     3-star      Vietnam                          4.427479
4     3-star  Netherlands                          4.415587


### 6.3. Feature 3: Điểm trung bình các tiêu chí chi tiết (Avg_Sub_Ratings)


Mục đích phục vụ Hỏi Đáp: Cung cấp điểm số định lượng nền tảng trên 6 khía cạnh dịch vụ cốt lõi (`Value`, `Rooms`, `Location`, `Cleanliness`, `Service`, `Sleep_Quality`) để Chatbot trả lời ngay các câu hỏi tổng quan theo từng khía cạnh cụ thể của khách sạn.



In [93]:
# df already in memory

sub_rating_cols = ["Value", "Rooms", "Location", "Cleanliness", "Service", "Sleep_Quality"]

feature_name = "Avg_Sub_Ratings"
df[feature_name] = df[sub_rating_cols].mean(axis=1, skipna=True)

# df state is kept in memory
print(f"Feature 3: {feature_name}")
print(df[sub_rating_cols + [feature_name]].head())


Feature 3: Avg_Sub_Ratings
   Value  Rooms  Location  Cleanliness  Service  Sleep_Quality  \
0    5.0    4.0       5.0          4.0      5.0            4.0   
1    4.0    4.0       4.0          4.0      4.0            4.0   
2    5.0    4.0       5.0          4.0      5.0            4.0   
3    5.0    4.0       5.0          4.0      5.0            4.0   
4    5.0    4.0       5.0          4.0      5.0            4.0   

   Avg_Sub_Ratings  
0              4.5  
1              4.0  
2              4.5  
3              4.5  
4              4.5  


### 6.4. Feature 4: Điểm trung bình theo Tháng & Tỉnh (mùa vụ  hotel_province_month_str_Avg_Score)


Mục đích phục vụ Hỏi Đáp: Cung cấp tri thức mùa vụ để Chatbot giải đáp thắc mắc về thời điểm du lịch (ví dụ: *'Đi vào mùa hè/tháng 7 khách sạn này có bị quá tải không, dịch vụ có bị giảm sút không?'*).



In [94]:
# df already in memory

group_col_1 = "hotel_province"
group_col_2 = "month_str"
value_col = "normalized_score"

feature_name = f"{group_col_1}_{group_col_2}_Avg_Score"
df[feature_name] = df.groupby([group_col_1, group_col_2])[value_col].transform('mean')

# df state is kept in memory
print(f"Feature 4: {feature_name}")
print(df[[group_col_1, group_col_2, feature_name]].head())


Feature 4: hotel_province_month_str_Avg_Score
  hotel_province month_str  hotel_province_month_str_Avg_Score
0       Hà Nội       Jul                            4.696245
1       Hà Nội       Apr                            4.604385
2       Hà Nội       May                            4.676811
3       Hà Nội       Apr                            4.604385
4       Hà Nội       Dec                            4.644784


### 6.5. Feature 5: Độ dài nội dung review (Content_Length_Chars & Content_Length_Words)


Mục đích phục vụ Hỏi Đáp: Đo lường độ chi tiết của bài đánh giá (Review Informativeness). Các câu trả lời của Chatbot sẽ ưu tiên trích dẫn bằng chứng từ các review có độ dài và chiều sâu thông tin thay vì các bài nhận xét quá ngắn ngủn.



In [95]:
# df already in memory

df["normalized_content"] = df["normalized_content"].fillna("").astype(str)

feature_name = "Content_Length_Chars"
df[feature_name] = df["normalized_content"].apply(len)

# df state is kept in memory
print(f"Feature 5: {feature_name}")
print(df[["Word_count", "normalized_content", feature_name]].head())


Feature 5: Content_Length_Chars
   Word_count                                 normalized_content  \
0          44  Good hotel i have ever stayed in Vietnam, good...   
1         431  Este hotel está muy cerca del barrio de las em...   
2          71  This place was very nice. Our bedroom were cle...   
3          45  Đầy đủ dịch vụ tiện nghi Ăn sáng buffee ngon H...   
4          44  It was a amazing hotel. They helped very good ...   

   Content_Length_Chars  
0                   209  
1                  2456  
2                   359  
3                   203  
4                   222  


### 6.6. Feature 6: Tỉ lệ từ trên câu & mật độ nội dung (Avg_Word_Length)


Mục đích phục vụ Hỏi Đáp: Phản ánh mật độ từ vựng và chất lượng hành văn của review, hỗ trợ thuật toán lọc và xếp hạng các bài đánh giá có giá trị tham khảo cao nhất để đưa vào câu trả lời của Chatbot.



In [96]:
# df already in memory

df["normalized_content"] = df["normalized_content"].fillna("").astype(str)

def avg_word_length(text):
    words = text.split()
    if not words:
        return 0.0
    return sum(len(w) for w in words) / len(words)

feature_name = "Avg_Word_Length"
df[feature_name] = df["normalized_content"].apply(avg_word_length)

# df state is kept in memory
print(f"Feature 6: {feature_name}")
print(df[["normalized_content", feature_name]].head())


Feature 6: Avg_Word_Length
                                  normalized_content  Avg_Word_Length
0  Good hotel i have ever stayed in Vietnam, good...         3.772727
1  Este hotel está muy cerca del barrio de las em...         4.700696
2  This place was very nice. Our bedroom were cle...         4.070423
3  Đầy đủ dịch vụ tiện nghi Ăn sáng buffee ngon H...         3.533333
4  It was a amazing hotel. They helped very good ...         4.068182


### 6.7. Feature 7: Số từ khóa tích cực/tiêu cực trong review (Positive_Keyword_Count & Negative_Keyword_Count)


Mục đích phục vụ Hỏi Đáp: Đếm số lượng từ khóa tích cực và tiêu cực trong review, giúp Chatbot phát hiện nhanh các điểm khen/chê nổi bật để tổng hợp câu trả lời đa chiều và khách quan cho người dùng.



In [97]:
# df already in memory
df["normalized_content"] = df["normalized_content"].fillna("").astype(str)

# Bộ từ điển cảm xúc tiếng Anh mở rộng (và song ngữ hỗ trợ)
positive_words = {
    'great', 'excellent', 'amazing', 'clean', 'friendly', 'comfortable',
    'wonderful', 'perfect', 'lovely', 'helpful', 'nice', 'good', 'best',
    'outstanding', 'superb', 'exceptional', 'spacious', 'delicious', 'recommend',
    'tốt', 'tuyệt', 'sạch', 'thân', 'thiện', 'thoải', 'mái', 'hài', 'lòng', 'tuyệt vời'
}
negative_words = {
    'dirty', 'rude', 'terrible', 'bad', 'poor', 'awful', 'noisy',
    'smell', 'broken', 'worst', 'disappointing', 'uncomfortable', 'horrible',
    'cockroach', 'bug', 'mold', 'stain', 'overpriced', 'never', 'avoid',
    'tệ', 'bẩn', 'ồn', 'hỏng', 'kém', 'thất', 'vọng', 'khó', 'chịu'
}

def count_keywords(text, keyword_set):
    words = re.findall(r'\b\w+\b', text.lower())
    return sum(1 for w in words if w in keyword_set)

df["Positive_Keyword_Count"] = df["normalized_content"].apply(lambda x: count_keywords(x, positive_words))
df["Negative_Keyword_Count"] = df["normalized_content"].apply(lambda x: count_keywords(x, negative_words))

# df state is kept in memory
print("Feature 7: Positive_Keyword_Count & Negative_Keyword_Count")
print(df[["normalized_content", "Positive_Keyword_Count", "Negative_Keyword_Count"]].head())


Feature 7: Positive_Keyword_Count & Negative_Keyword_Count
                                  normalized_content  Positive_Keyword_Count  \
0  Good hotel i have ever stayed in Vietnam, good...                       6   
1  Este hotel está muy cerca del barrio de las em...                       0   
2  This place was very nice. Our bedroom were cle...                       6   
3  Đầy đủ dịch vụ tiện nghi Ăn sáng buffee ngon H...                       0   
4  It was a amazing hotel. They helped very good ...                       4   

   Negative_Keyword_Count  
0                       0  
1                       0  
2                       0  
3                       0  
4                       0  


### 6.8. Feature 8: Độ mới của review  số tháng tính từ review đến hiện tại (Review_Recency_Months)


Mục đích phục vụ Hỏi Đáp: Xác định độ mới của bài đánh giá (Review Recency). Chatbot ưu tiên các review gần đây nhất để đảm bảo thông tin về chất lượng phòng ốc, cách âm và dịch vụ phản ánh đúng hiện trạng thực tế của khách sạn.



In [98]:
# df already in memory

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m", errors="coerce")

reference_date = pd.to_datetime(datetime.now().strftime("%Y-%m"))

df["Review_Recency_Months"] = (
    (reference_date.year - df["Date"].dt.year) * 12
    + (reference_date.month - df["Date"].dt.month)
)

# df state is kept in memory
print("Feature 8: Review_Recency_Months")
print(df[["Date", "Review_Recency_Months"]].head())


Feature 8: Review_Recency_Months
        Date  Review_Recency_Months
0 2023-07-01                     38
1 2023-04-01                     41
2 2023-05-01                     40
3 2023-04-01                     41
4 2022-12-01                     45


### 6.9. Feature 9: Nhận diện & Mã hóa Ngôn ngữ Đánh giá (Review_Language_Code & Is_Original_English)



Mục đích phục vụ Hỏi Đáp: Chuẩn hóa mã ngôn ngữ review (`Review_Language_Code`) phục vụ truy vấn và trả lời câu hỏi đa ngôn ngữ (Native Multilingual QA), hỗ trợ du khách hỏi và nhận câu trả lời bằng tiếng Việt, tiếng Anh hoặc ngôn ngữ bản địa của họ.



In [99]:
# df already in memory

# 1. Mã hóa cờ ngôn ngữ tiếng Anh (để tương thích ngược với các downstream models)
feature_name = "Is_Original_English"
df[feature_name] = (df["language_code"].astype(str).str.lower() == "en").astype(int)

# 2. Chuẩn hóa mã ngôn ngữ review phục vụ Multilingual Embeddings
df["Review_Language_Code"] = df["language_code"].fillna("unknown").astype(str).str.lower()

# df state is kept in memory
print(f"Feature 9: {feature_name} & Review_Language_Code")
print(df[["language_code", "Review_Language_Code", feature_name]].head(10))
print(f"\nPhân bố ngôn ngữ trong features.jsonl:")
print(df["Review_Language_Code"].value_counts().head(10))


Feature 9: Is_Original_English & Review_Language_Code
  language_code Review_Language_Code  Is_Original_English
0            en                   en                    1
1            es                   es                    0
2            en                   en                    1
3            vi                   vi                    0
4            en                   en                    1
5            en                   en                    1
6            ko                   ko                    0
7         zh-tw                zh-tw                    0
8            vi                   vi                    0
9            vi                   vi                    0

Phân bố ngôn ngữ trong features.jsonl:
Review_Language_Code
en    527335
vi     48396
ko     42547
fr     38770
de     25736
ru     22612
ja     18552
es     14517
nl     11359
it      9020
Name: count, dtype: int64


### 6.10. Feature 10: Số lượng dấu câu thể hiện cảm xúc mạnh (Emotion_Punctuation_Count)


Mục đích phục vụ Hỏi Đáp: Nhận diện các review chứa cảm xúc mạnh (khen ngợi đặc biệt hoặc phàn nàn gay gắt qua dấu `!` và `?`), giúp Chatbot chú ý đưa ra cảnh báo hoặc lưu ý chân thực cho khách hàng.



In [100]:
# df already in memory
df["normalized_content"] = df["normalized_content"].fillna("").astype(str)

def count_emotion_punctuation(text):
    exclamation = text.count("!")
    question = text.count("?")
    return exclamation + question

feature_name = "Emotion_Punctuation_Count"
df[feature_name] = df["normalized_content"].apply(count_emotion_punctuation)

# df state is kept in memory
print(f"Feature 10: {feature_name}")
print(df[["normalized_content", feature_name]].head())


Feature 10: Emotion_Punctuation_Count
                                  normalized_content  \
0  Good hotel i have ever stayed in Vietnam, good...   
1  Este hotel está muy cerca del barrio de las em...   
2  This place was very nice. Our bedroom were cle...   
3  Đầy đủ dịch vụ tiện nghi Ăn sáng buffee ngon H...   
4  It was a amazing hotel. They helped very good ...   

   Emotion_Punctuation_Count  
0                          0  
1                          0  
2                          0  
3                          0  
4                          1  


### 6.11. Xây dựng Cơ Sở Dữ Liệu Quan Hệ (Relational DB) & Xuất Dữ Liệu Huấn Luyện Chatbot

Chuyển đổi dữ liệu phẳng (flat table) đã tích hợp đầy đủ 10 features thành 3 bảng quan hệ trong SQLite (`chatbot.db`) và xuất dữ liệu `reviews_qa.jsonl` chuẩn format phục vụ huấn luyện Chatbot:
1. `hotels`: Metadata khách sạn, điểm trung bình và xếp hạng các tiêu chí.
2. `reviews`: Nội dung bài đánh giá cùng toàn bộ 10 features đã trích xuất, liên kết với khách sạn qua `hotel_id`.
3. `hotel_summaries`: Bảng tóm tắt theo từng khía cạnh dịch vụ (`aspect`).


In [101]:
# =====================================================================
# 6.11. Xây dựng Cơ Sở Dữ Liệu SQLite & Xuất Dữ Liệu Huấn Luyện Chatbot
# =====================================================================



def create_hotels_table(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo bảng HOTELS (metadata khách sạn + điểm trung bình các tiêu chí)."""
    print(f"\n Tạo bảng HOTELS...")
    hotel_records = []

    grouped = df.groupby(["hotel_name", "hotel_address"])
    for (name, address), group in tqdm(grouped, desc="   Processing hotels"):
        hotel_id = generate_hotel_id(name, address)
        province = group["hotel_province"].iloc[0]
        region = cfg.PROVINCE_TO_REGION.get(province, "Khác")
        star = group["hotel_star"].iloc[0] if "hotel_star" in group else None

        avg_score = round(group["normalized_score"].mean(), 2)
        total_reviews = len(group)

        record = {
            "hotel_id": hotel_id,
            "hotel_name": name,
            "hotel_address": address,
            "hotel_province": province,
            "hotel_region": region,
            "hotel_star": star,
            "avg_score": avg_score,
            "total_reviews": total_reviews,
        }

        for aspect in cfg.ASPECT_COLUMNS:
            if aspect in group.columns:
                valid_scores = group[aspect].dropna()
                record[f"avg_{aspect.lower()}"] = (
                    round(valid_scores.mean(), 2) if len(valid_scores) > 0 else None
                )
            else:
                record[f"avg_{aspect.lower()}"] = None

        hotel_records.append(record)

    hotels_df = pd.DataFrame(hotel_records)
    print(f"  [OK] Tạo xong {len(hotels_df):,} khách sạn")
    return hotels_df


def create_reviews_table(df: pd.DataFrame, hotels_df: pd.DataFrame) -> pd.DataFrame:
    """Tạo bảng REVIEWS  thêm hotel_id, review_id + toàn bộ 10 features."""
    print(f"\n[INFO] Tạo bảng REVIEWS ({len(df):,} reviews)...")

    df = df.copy()
    if "hotel_id" not in df.columns or df["hotel_id"].isnull().any():
        hotel_id_map = {}
        for _, row in hotels_df.iterrows():
            key = (row["hotel_name"], row["hotel_address"])
            hotel_id_map[key] = row["hotel_id"]
        df["hotel_id"] = [
            hotel_id_map.get((n, a), None)
            for n, a in zip(df["hotel_name"], df["hotel_address"])
        ]
    if "review_id" not in df.columns:
        df["review_id"] = range(1, len(df) + 1)
    if "hotel_region" not in df.columns:
        df["hotel_region"] = df["hotel_province"].map(cfg.PROVINCE_TO_REGION).fillna("Khác")

    review_columns = [
        "review_id",
        "hotel_id",
        "tripadvisor_hotel_id",
        "hotel_name",
        "hotel_address",
        "link",
        "id_url",
        "Date",
        "month",
        "year",
        "month_str",
        "normalized_score",
        "trip_type",
        "hotel_province",
        "hotel_region",
        "hotel_star",
        "Value",
        "Rooms",
        "Location",
        "Cleanliness",
        "Service",
        "Sleep_Quality",
        "normalized_content",
        "normalized_title",
        "Word_count",
        "language_code",
        "language",
        "nationality",
    ]

    feature_columns = [
        "hotel_province_trip_type_Avg_Score",
        "hotel_star_nationality_Avg_Score",
        "Avg_Sub_Ratings",
        "hotel_province_month_str_Avg_Score",
        "Content_Length_Chars",
        "Avg_Word_Length",
        "Positive_Keyword_Count",
        "Negative_Keyword_Count",
        "Review_Recency_Months",
        "Is_Original_English",
        "Emotion_Punctuation_Count",
    ]

    existing_cols = [c for c in review_columns + feature_columns if c in df.columns]
    reviews_df = df[existing_cols].copy()
    print(f"  [OK] Tạo xong {len(reviews_df):,} reviews ({len(existing_cols)} cột)")
    return reviews_df


def _extract_top_keywords(texts: pd.Series, keyword_list: list, top_n: int = 10) -> str:
    all_text = " ".join(texts.dropna().astype(str)).lower()
    counts = {}
    for kw in keyword_list:
        n = len(re.findall(r"\b" + re.escape(kw) + r"\b", all_text))
        if n > 0:
            counts[kw] = n
    sorted_kw = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return ", ".join(f"{kw} ({n})" for kw, n in sorted_kw) if sorted_kw else "N/A"


def create_hotel_summaries(reviews_df: pd.DataFrame, hotels_df: pd.DataFrame) -> pd.DataFrame:
    """Tạo bảng HOTEL_SUMMARIES theo aspect."""
    print(f"\n[STATS] Tạo bảng HOTEL_SUMMARIES...")
    summaries = []
    hotel_reviews = reviews_df.groupby("hotel_id")

    for hotel_id, group in tqdm(hotel_reviews, desc="   Processing summaries"):
        for aspect in cfg.ASPECT_COLUMNS:
            if aspect not in group.columns:
                continue
            aspect_scores = group[aspect].dropna()
            if len(aspect_scores) == 0:
                continue

            pos_reviews = group[group["normalized_score"] >= 4]["normalized_content"]
            neg_reviews = group[group["normalized_score"] <= 2]["normalized_content"]

            summaries.append({
                "hotel_id": hotel_id,
                "aspect": aspect,
                "aspect_vi": cfg.ASPECT_LABELS_VI.get(aspect, aspect),
                "avg_aspect_score": round(aspect_scores.mean(), 2),
                "review_count": len(aspect_scores),
                "positive_mentions": _extract_top_keywords(pos_reviews, cfg.POSITIVE_KEYWORDS, top_n=5),
                "negative_mentions": _extract_top_keywords(neg_reviews, cfg.NEGATIVE_KEYWORDS, top_n=5),
            })

    summaries_df = pd.DataFrame(summaries)
    print(f"  [OK] Tạo xong {len(summaries_df):,} hotel summaries")
    return summaries_df


def save_to_sqlite(hotels_df: pd.DataFrame, reviews_df: pd.DataFrame, summaries_df: pd.DataFrame, db_path: str = None) -> None:
    """Lưu 3 bảng vào SQLite database."""
    db_path = db_path or cfg.NEW_DB_PATH
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    print(f"\n Lưu database: {db_path}...")

    conn = sqlite3.connect(db_path)
    hotels_df.to_sql("hotels", conn, if_exists="replace", index=False)
    reviews_df.to_sql("reviews", conn, if_exists="replace", index=False)
    summaries_df.to_sql("hotel_summaries", conn, if_exists="replace", index=False)

    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_hotels_province ON hotels(hotel_province)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_hotels_region ON hotels(hotel_region)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_hotels_star ON hotels(hotel_star)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_reviews_hotel_id ON reviews(hotel_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_reviews_score ON reviews(normalized_score)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_summaries_hotel_id ON hotel_summaries(hotel_id)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_summaries_aspect ON hotel_summaries(aspect)")
    conn.commit()
    conn.close()

    size_mb = os.path.getsize(db_path) / (1024 * 1024)
    print(f"  [OK] Đã lưu {db_path} ({size_mb:.1f} MB)")


def export_for_training(reviews_df: pd.DataFrame, hotels_df: pd.DataFrame) -> None:
    """Export dữ liệu training cho NLP/Chatbot."""
    os.makedirs(cfg.TRAINING_DIR, exist_ok=True)
    print(f"\n Export dữ liệu training  {cfg.TRAINING_DIR}")

    jsonl_path = os.path.join(cfg.TRAINING_DIR, "reviews_qa.jsonl")
    hotel_info_map = hotels_df.set_index("hotel_id").to_dict("index")

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in tqdm(reviews_df.iterrows(), total=len(reviews_df), desc="   Writing JSONL"):
            hotel_id = row.get("hotel_id")
            hotel_info = hotel_info_map.get(hotel_id, {})

            record = {
                "hotel_name": hotel_info.get("hotel_name", ""),
                "hotel_province": row.get("hotel_province", ""),
                "hotel_region": row.get("hotel_region", ""),
                "hotel_star": hotel_info.get("hotel_star", ""),
                "hotel_address": hotel_info.get("hotel_address", ""),
                "avg_hotel_score": hotel_info.get("avg_score"),
                "review_title": row.get("normalized_title", ""),
                "review_content": row.get("normalized_content", ""),
                "review_score": row.get("normalized_score"),
                "language": row.get("language", ""),
                "language_code": row.get("language_code", ""),
                "trip_type": row.get("trip_type", ""),
                "nationality": row.get("nationality", ""),
                "aspects": {
                    aspect: row.get(aspect)
                    for aspect in cfg.ASPECT_COLUMNS
                    if pd.notna(row.get(aspect))
                },
                "features": {
                    "avg_sub_ratings": row.get("Avg_Sub_Ratings"),
                    "positive_keywords": row.get("Positive_Keyword_Count"),
                    "negative_keywords": row.get("Negative_Keyword_Count"),
                    "review_recency_months": row.get("Review_Recency_Months"),
                    "is_original_english": row.get("Is_Original_English"),
                    "emotion_punctuation_count": row.get("Emotion_Punctuation_Count"),
                },
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    size_mb = os.path.getsize(jsonl_path) / (1024 * 1024)
    print(f"  [OK] reviews_qa.jsonl: {len(reviews_df):,} records ({size_mb:.1f} MB)")

    # Hotels summary CSV
    jsonl_hotels_path = os.path.join(cfg.TRAINING_DIR, "hotels_summary.jsonl")
    hotels_df.to_json(jsonl_hotels_path, orient="records", lines=True, force_ascii=False)
    print(f"  [OK] hotels_summary.jsonl: {len(hotels_df):,} hotels")

    # Stats JSON
    stats = {
        "export_time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "total_reviews": len(reviews_df),
        "total_hotels": len(hotels_df),
        "total_provinces": int(reviews_df["hotel_province"].nunique()),
        "score_distribution": {
            str(k): int(v) for k, v in reviews_df["normalized_score"].value_counts().sort_index().items()
        },
        "config_used": {
            "language_strategy": cfg.LANGUAGE_STRATEGY,
            "min_word_count": cfg.MIN_WORD_COUNT,
            "max_word_count": cfg.MAX_WORD_COUNT,
            "min_hotel_reviews": cfg.MIN_HOTEL_REVIEWS,
        },
    }
    stats_path = os.path.join(cfg.TRAINING_DIR, "data_stats.json")
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)
    print(f"  [OK] data_stats.json: thống kê training data")


def run_chatbot_db_pipeline(df_final):
    """Chạy toàn bộ quá trình tạo DB và xuất training data từ dataframe."""
    t_start = time.time()
    features_path = os.path.join(cfg.TRAINING_DIR, "features.jsonl")
    
    print(f"\n Cập nhật features.jsonl ra {features_path} để tracking...")
    # Lưu file features.jsonl cho mục đích tracking
    export_df_to_jsonl(df_final, features_path)

    hotels_df = create_hotels_table(df_final)
    reviews_df = create_reviews_table(df_final, hotels_df)
    summaries_df = create_hotel_summaries(reviews_df, hotels_df)

    save_to_sqlite(hotels_df, reviews_df, summaries_df)
    export_for_training(reviews_df, hotels_df)
    print(f"\n[TIME]  Tổng thời gian tạo DB & Export: {time.time() - t_start:.1f}s")
    print(" Hoàn tất xây dựng Database và xuất dữ liệu Chatbot!")


# Thực thi xây dựng Database và xuất dữ liệu từ df (đã chạy qua 10 bước feature)
run_chatbot_db_pipeline(df)




 Cập nhật features.jsonl ra /Users/trietnguyen/Documents/Hotel-Review-AI-Chatbot/Data/train/features.jsonl để tracking...


/var/folders/m_/115k21gx4c12qm05_3nsm0hw0000gn/T/ipykernel_1852/183401090.py:63: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  chunk_json = chunk.to_json(orient="records", lines=True, force_ascii=False)
/var/folders/m_/115k21gx4c12qm05_3nsm0hw0000gn/T/ipykernel_1852/183401090.py:63: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  chunk_json = chunk.to_json(orient="records", lines=True, force_ascii=False)
/var/folders/m_/115k21gx4c12qm05_3nsm0hw0000gn/T/ipykernel_1852/183401090.py:63: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  chunk_json = chunk.to_json(orient="records", lines=True, force_ascii=False)
/var/folders/m_/115k21gx4c12qm05_3nsm0hw0000gn/T/ipykernel_1852/183401090.py:63: Pandas4Warning: The


 Tạo bảng HOTELS...


   Processing hotels: 100%|██████████| 12658/12658 [00:19<00:00, 662.80it/s]


  [OK] Tạo xong 12,658 khách sạn

[INFO] Tạo bảng REVIEWS (782,580 reviews)...
  [OK] Tạo xong 782,580 reviews (38 cột)

[STATS] Tạo bảng HOTEL_SUMMARIES...


   Processing summaries: 100%|██████████| 12658/12658 [20:03<00:00, 10.52it/s] 


  [OK] Tạo xong 75,948 hotel summaries

 Lưu database: /Users/trietnguyen/Documents/Hotel-Review-AI-Chatbot/Data/db/chatbot.db...
  [OK] Đã lưu /Users/trietnguyen/Documents/Hotel-Review-AI-Chatbot/Data/db/chatbot.db (2601.0 MB)

 Export dữ liệu training  /Users/trietnguyen/Documents/Hotel-Review-AI-Chatbot/Data/train


   Writing JSONL: 100%|██████████| 782580/782580 [01:49<00:00, 7127.20it/s]

  [OK] reviews_qa.jsonl: 782,580 records (918.4 MB)
  [OK] hotels_summary.jsonl: 12,658 hotels
  [OK] data_stats.json: thống kê training data

[TIME]  Tổng thời gian tạo DB & Export: 1398.7s
 Hoàn tất xây dựng Database và xuất dữ liệu Chatbot!


## 7. Xây dựng và Huấn luyện Mô hình

Phần này định nghĩa dữ liệu đầu vào, kiến trúc mô hình (CRAM-ABSA) và quá trình huấn luyện.


In [ ]:
ASPECT_CATEGORIES = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
SENTIMENT_POLARITIES = ["Positive", "Negative", "Neutral"]

def extract_triplets_and_meta(item: Dict[str, Any]) -> Dict[str, Any] | None:
    """Trích xuất text, triplets, rating, hotel info từ 1 record Label Studio."""
    text = item.get("data", {}).get("text", "")
    if not text or not text.strip():
        return None

    meta = item.get("data", {}).get("meta_info", {})
    hotel_name = meta.get("hotel name", "").strip()
    hotel_address = meta.get("hotel address", "").strip()
    score_str = meta.get("score", "")

    try:
        score = float(score_str)
    except (ValueError, TypeError):
        score = 0.0

    anns = item.get("annotations", [])
    if not anns or not anns[0].get("result"):
        return None

    results = anns[0]["result"]
    span_map = defaultdict(lambda: {"aspects": [], "sentiments": [], "text": ""})

    for res in results:
        val = res.get("value", {})
        start = val.get("start")
        end = val.get("end")
        labels = val.get("labels", [])
        span_text = val.get("text", "")

        if start is None or end is None or not labels:
            continue

        key = (start, end)
        span_map[key]["text"] = span_text
        for lbl in labels:
            if lbl in ASPECT_CATEGORIES:
                span_map[key]["aspects"].append(lbl)
            elif lbl in SENTIMENT_POLARITIES:
                span_map[key]["sentiments"].append(lbl)

    triplets = []
    for (start, end), data in span_map.items():
        aspects = data["aspects"]
        sentiments = data["sentiments"]
        span_text = data["text"]

        if not aspects:
            continue

        # Nếu thiếu sentiment, gán mặc định Positive
        if not sentiments:
            sentiments = ["Positive"]

        for asp in aspects:
            for sent in sentiments:
                triplets.append({
                    "start": start,
                    "end": end,
                    "aspect": asp,
                    "sentiment": sent,
                    "span_text": span_text
                })

    if not triplets:
        return None

    # Xác định mức độ mâu thuẫn (Conflict Flag)
    sentiments_present = set(t["sentiment"] for t in triplets)
    has_mixed_sentiment = ("Positive" in sentiments_present and "Negative" in sentiments_present)
    high_rating_with_negative = (score >= 4.0 and "Negative" in sentiments_present)
    low_rating_with_positive = (score <= 2.0 and "Positive" in sentiments_present)

    is_conflict = has_mixed_sentiment or high_rating_with_negative or low_rating_with_positive

    return {
        "id": item.get("id"),
        "text": text,
        "hotel_name": hotel_name,
        "hotel_address": hotel_address,
        "hotel_id": f"{hotel_name.lower()}___{hotel_address.lower()}",
        "rating": score,
        "triplets": triplets,
        "is_conflict": is_conflict,
        "conflict_reasons": {
            "has_mixed_sentiment": has_mixed_sentiment,
            "high_rating_with_negative": high_rating_with_negative,
            "low_rating_with_positive": low_rating_with_positive
        }
    }

def prepare_hotel_disjoint_splits(
    raw_path: str = "Data/dts_raw/TripAdvisor_EN.json",
    output_dir: str = "Data/split",
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    seed: int = 42
) -> Dict[str, int]:
    """Chia dataset theo khách sạn (Hotel-Disjoint Split) với 0% overlap."""
    print(f"Loading raw dataset from {raw_path}...")
    with open(raw_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    parsed_records = []
    hotel_to_records = defaultdict(list)

    for item in raw_data:
        rec = extract_triplets_and_meta(item)
        if rec is not None:
            parsed_records.append(rec)
            hotel_to_records[rec["hotel_id"]].append(rec)

    total_records = len(parsed_records)
    total_hotels = len(hotel_to_records)
    print(f"Total valid annotated records: {total_records:,} across {total_hotels:,} hotels.")

    # Xáo trộn danh sách khách sạn
    random.seed(seed)
    hotel_ids = list(hotel_to_records.keys())
    random.shuffle(hotel_ids)

    train_target = int(total_records * train_ratio)
    val_target = int(total_records * val_ratio)

    train_records = []
    val_records = []
    test_records = []

    train_hotels = set()
    val_hotels = set()
    test_hotels = set()

    for h_id in hotel_ids:
        records = hotel_to_records[h_id]
        if len(train_records) < train_target:
            train_records.extend(records)
            train_hotels.add(h_id)
        elif len(val_records) < val_target:
            val_records.extend(records)
            val_hotels.add(h_id)
        else:
            test_records.extend(records)
            test_hotels.add(h_id)

    # Kiểm tra tính toàn vẹn 0% hotel overlap
    overlap_train_val = train_hotels.intersection(val_hotels)
    overlap_train_test = train_hotels.intersection(test_hotels)
    overlap_val_test = val_hotels.intersection(test_hotels)

    assert len(overlap_train_val) == 0, f"Error: Train/Val hotel overlap: {len(overlap_train_val)}"
    assert len(overlap_train_test) == 0, f"Error: Train/Test hotel overlap: {len(overlap_train_test)}"
    assert len(overlap_val_test) == 0, f"Error: Val/Test hotel overlap: {len(overlap_val_test)}"

    # Tách test set thành consistent vs conflict
    test_conflict = [r for r in test_records if r["is_conflict"]]
    test_consistent = [r for r in test_records if not r["is_conflict"]]

    os.makedirs(output_dir, exist_ok=True)

    def save_jsonl(filepath: str, data: List[Dict[str, Any]]):
        with open(filepath, "w", encoding="utf-8") as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    train_path = os.path.join(output_dir, "train.jsonl")
    val_path = os.path.join(output_dir, "val.jsonl")
    test_path = os.path.join(output_dir, "test.jsonl")
    conflict_path = os.path.join(output_dir, "test_conflict.jsonl")
    consistent_path = os.path.join(output_dir, "test_consistent.jsonl")

    save_jsonl(train_path, train_records)
    save_jsonl(val_path, val_records)
    save_jsonl(test_path, test_records)
    save_jsonl(conflict_path, test_conflict)
    save_jsonl(consistent_path, test_consistent)

    stats = {
        "total_records": total_records,
        "total_hotels": total_hotels,
        "train_reviews": len(train_records),
        "train_hotels": len(train_hotels),
        "val_reviews": len(val_records),
        "val_hotels": len(val_hotels),
        "test_reviews": len(test_records),
        "test_hotels": len(test_hotels),
        "test_conflict_reviews": len(test_conflict),
        "test_consistent_reviews": len(test_consistent)
    }

    stats_path = os.path.join(output_dir, "split_stats.json")
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 60)
    print(" HOTEL-DISJOINT SPLIT THÀNH CÔNG (0% HOTEL OVERLAP):")
    print("=" * 60)
    print(f" Train set : {len(train_records):>5,} reviews ({len(train_hotels):>4,} hotels) -> {train_path}")
    print(f" Val set   : {len(val_records):>5,} reviews ({len(val_hotels):>4,} hotels) -> {val_path}")
    print(f" Test set  : {len(test_records):>5,} reviews ({len(test_hotels):>4,} hotels) -> {test_path}")
    print(f"   |- Conflict Subset   : {len(test_conflict):>5,} reviews -> {conflict_path}")
    print(f"   |- Consistent Subset : {len(test_consistent):>5,} reviews -> {consistent_path}")
    print("=" * 60 + "\n")

    return stats

if __name__ == "__main__":
    prepare_hotel_disjoint_splits()



Loading raw dataset from Data/dts_raw/TripAdvisor_EN.json...
Total valid annotated records: 9,867 across 2,610 hotels.

 HOTEL-DISJOINT SPLIT THÀNH CÔNG (0% HOTEL OVERLAP):
 Train set : 6,909 reviews (1,815 hotels) -> Data/split/train.jsonl
 Val set   : 1,484 reviews ( 410 hotels) -> Data/split/val.jsonl
 Test set  : 1,474 reviews ( 385 hotels) -> Data/split/test.jsonl
   |- Conflict Subset   :   236 reviews -> Data/split/test_conflict.jsonl
   |- Consistent Subset : 1,238 reviews -> Data/split/test_consistent.jsonl



### 7.2. Định nghĩa Mô hình (Model Definition)

Định nghĩa cấu trúc mô hình CRAM-ABSA dùng cho bài toán trích xuất khía cạnh (Aspect) và dự đoán đánh giá (Rating).


In [3]:
# =====================================================================
# 1. Định Nghĩa Không Gian Nhãn BIO Cho Triplet Extraction
# =====================================================================
ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
SENTIMENTS = ["Positive", "Negative", "Neutral"]

def build_label_vocab() -> Tuple[Dict[str, int], Dict[int, str]]:
    """Tạo từ điển BIO cho (Aspect, Sentiment): e.g. B-Service-Positive, I-Service-Positive."""
    labels = ["O"]
    for asp in ASPECTS:
        for sent in SENTIMENTS:
            labels.append(f"B-{asp}-{sent}")
            labels.append(f"I-{asp}-{sent}")

    label2id = {lbl: idx for idx, lbl in enumerate(labels)}
    id2label = {idx: lbl for idx, lbl in enumerate(labels)}
    return label2id, id2label

LABEL2ID, ID2LABEL = build_label_vocab()
NUM_EXTRACTION_LABELS = len(LABEL2ID)  # 37 classes

# =====================================================================
# 2. Focal Loss Cho Trích Xuất Ý Kiến
# =====================================================================
class FocalLoss(nn.Module):
    """
    Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    Giảm trọng số các mẫu dễ (nhãn 'O' chiếm đa số), tập trung vào các aspect hiếm.
    """
    def __init__(self, gamma: float = 2.0, alpha: Optional[torch.Tensor] = None, ignore_index: int = -100):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits: (B, L, C), targets: (B, L)
        active_mask = (targets != self.ignore_index)
        if not active_mask.any():
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        logits_flat = logits[active_mask]      # (N, C)
        targets_flat = targets[active_mask]    # (N,)

        ce_loss = F.cross_entropy(logits_flat, targets_flat, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = ((1.0 - p_t) ** self.gamma) * ce_loss

        if self.alpha is not None:
            alpha_t = self.alpha[targets_flat]
            focal_loss = alpha_t * focal_loss

        return focal_loss.mean()

# =====================================================================
# 3. Attention Pooling Layer
# =====================================================================
class AttentionPooling(nn.Module):
    """Học trọng số tự chú ý trên các hidden states để nén chuỗi thành vector văn bản."""
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.query = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        # hidden_states: (B, L, D), attention_mask: (B, L)
        scores = self.query(hidden_states).squeeze(-1)  # (B, L)
        scores = scores.masked_fill(attention_mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1).unsqueeze(-1)  # (B, L, 1)
        pooled = torch.sum(hidden_states * weights, dim=1)  # (B, D)
        return pooled

# =====================================================================
# 4. Conflict Attenuation Gating Layer
# =====================================================================
class ConflictAttenuationGate(nn.Module):
    """
    Cơ chế cổng gating điều tiết thông tin:
    Học trọng số suy giảm gradient / biểu diễn toàn cục khi phát hiện xung đột
    giữa các khía cạnh cục bộ (mixed sentiment) và rating tổng thể.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.gate_dense = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid()
        )
        self.transform = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )

    def forward(self, token_hidden: torch.Tensor, global_pooled: torch.Tensor) -> torch.Tensor:
        # token_hidden: (B, L, D), global_pooled: (B, D)
        B, L, D = token_hidden.size()
        global_expanded = global_pooled.unsqueeze(1).expand(B, L, D)  # (B, L, D)
        
        combined = torch.cat([token_hidden, global_expanded], dim=-1)  # (B, L, 2D)
        gate = self.gate_dense(combined)  # (B, L, D) in [0, 1]
        
        # Gated modulation: nếu mâu thuẫn lớn, gate sẽ suy giảm ảnh hưởng của global_pooled
        modulated = token_hidden + gate * self.transform(global_expanded)
        return modulated

# =====================================================================
# 5. Proposed Model: CRAM_ABSA
# =====================================================================
class CRAM_ABSA(nn.Module):
    """
    Conflict-attenuated Representation for Aspect Multi-tasking (CRAM-ABSA)
    Kết hợp Opinion Extraction + Continuous Rating Prediction + Conflict Gate.
    """
    def __init__(
        self,
        model_name_or_path: str = "microsoft/deberta-v3-base",
        num_labels: int = NUM_EXTRACTION_LABELS,
        dropout_prob: float = 0.2,
        focal_gamma: float = 2.0
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name_or_path)
        self.encoder = AutoModel.from_pretrained(model_name_or_path, config=self.config)
        hidden_dim = self.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)

        # 1. Attention Pooling
        self.attention_pooling = AttentionPooling(hidden_dim)

        # 2. Continuous Rating Head (dự đoán điểm 1.0 - 5.0)
        self.rating_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim // 2, 1)
        )

        # 3. Conflict Attenuation Gating
        self.conflict_gate = ConflictAttenuationGate(hidden_dim)

        # 4. Extraction Head (Sequence Labeling)
        self.extraction_head = nn.Linear(hidden_dim, num_labels)

        # 5. Loss Functions
        self.focal_loss_fn = FocalLoss(gamma=focal_gamma)
        self.rating_loss_fn = nn.SmoothL1Loss()

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: Optional[torch.Tensor] = None,
        extraction_labels: Optional[torch.Tensor] = None,
        rating_labels: Optional[torch.Tensor] = None,
        lambda_ext: float = 1.0,
        lambda_rate: float = 0.5
    ) -> Dict[str, torch.Tensor]:

        # 1. Shared Transformer Encoder
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None and hasattr(self.config, "type_vocab_size") and self.config.type_vocab_size > 0:
            encoder_kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(**encoder_kwargs)
        sequence_output = outputs.last_hidden_state  # (B, L, D)

        # 2. Attention Pooling -> Global Review Representation
        global_repr = self.attention_pooling(sequence_output, attention_mask)  # (B, D)

        # 3. Continuous Rating Prediction
        rating_preds = self.rating_head(global_repr).squeeze(-1)  # (B,)

        # 4. Conflict Attenuation Gating
        gated_sequence = self.conflict_gate(sequence_output, global_repr)  # (B, L, D)
        gated_sequence = self.dropout(gated_sequence)

        # 5. Extraction Head Logits
        extraction_logits = self.extraction_head(gated_sequence)  # (B, L, Num_labels)

        result = {
            "extraction_logits": extraction_logits,
            "rating_preds": rating_preds,
            "global_repr": global_repr
        }

        # 6. Tính toán Multi-Task Loss nếu có labels
        loss = None
        if extraction_labels is not None and rating_labels is not None:
            ext_loss = self.focal_loss_fn(extraction_logits, extraction_labels)
            rate_loss = self.rating_loss_fn(rating_preds, rating_labels.float())
            total_loss = (lambda_ext * ext_loss) + (lambda_rate * rate_loss)

            result.update({
                "loss": total_loss,
                "extraction_loss": ext_loss,
                "rating_loss": rate_loss
            })

        return result

# =====================================================================
# 6. Baseline Model: PlainMTL_ABSA
# =====================================================================
class PlainMTL_ABSA(nn.Module):
    """
    Plain Multi-Task Learning Baseline:
    Shared Transformer Encoder + Extraction Head (CE) + Mean Pooling + Rating Head.
    Không có Conflict Gate và Không có Focal Loss.
    """
    def __init__(
        self,
        model_name_or_path: str = "microsoft/deberta-v3-base",
        num_labels: int = NUM_EXTRACTION_LABELS,
        dropout_prob: float = 0.2
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name_or_path)
        self.encoder = AutoModel.from_pretrained(model_name_or_path, config=self.config)
        hidden_dim = self.config.hidden_size

        self.dropout = nn.Dropout(dropout_prob)

        # Standard Linear Extraction Head
        self.extraction_head = nn.Linear(hidden_dim, num_labels)

        # Standard Linear Rating Head
        self.rating_head = nn.Linear(hidden_dim, 1)

        self.ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
        self.rating_loss_fn = nn.MSELoss()

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: Optional[torch.Tensor] = None,
        extraction_labels: Optional[torch.Tensor] = None,
        rating_labels: Optional[torch.Tensor] = None,
        lambda_ext: float = 1.0,
        lambda_rate: float = 0.5
    ) -> Dict[str, torch.Tensor]:

        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None and hasattr(self.config, "type_vocab_size") and self.config.type_vocab_size > 0:
            encoder_kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(**encoder_kwargs)
        sequence_output = self.dropout(outputs.last_hidden_state)  # (B, L, D)

        # Mean Pooling
        mask_expanded = attention_mask.unsqueeze(-1).expand_as(sequence_output)
        sum_embeddings = torch.sum(sequence_output * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask  # (B, D)

        # Predictions
        extraction_logits = self.extraction_head(sequence_output)
        rating_preds = self.rating_head(mean_pooled).squeeze(-1)

        result = {
            "extraction_logits": extraction_logits,
            "rating_preds": rating_preds
        }

        if extraction_labels is not None and rating_labels is not None:
            B, L, C = extraction_logits.size()
            ext_loss = self.ce_loss_fn(extraction_logits.view(-1, C), extraction_labels.view(-1))
            rate_loss = self.rating_loss_fn(rating_preds, rating_labels.float())
            total_loss = (lambda_ext * ext_loss) + (lambda_rate * rate_loss)

            result.update({
                "loss": total_loss,
                "extraction_loss": ext_loss,
                "rating_loss": rate_loss
            })

        return result


ModuleNotFoundError: No module named 'torch'

### 7.3. Hàm Đánh giá Mô hình (Evaluation)

Hàm dùng để đánh giá mô hình trên tập validation và test trong quá trình huấn luyện.


In [ ]:
"""
modules/evaluate.py
===================
Đo lường và đánh giá toàn diện mô hình:
1. Extraction Metrics:
   - Exact Triplet Match F1 (Precision, Recall, F1)
   - Overlap Triplet Match F1 (IoU >= 0.5)
   - Per-aspect F1 breakdown (Service, Facility, Amenity, Experience, Loyalty, Branding)
2. Rating Metrics:
   - MAE, RMSE, Pearson Correlation, Within-1 Accuracy
3. Conflict Diagnostic:
   - Đánh giá riêng biệt trên Conflict Subset vs Consistent Subset
"""


def compute_span_iou(span_a: Tuple[int, int], span_b: Tuple[int, int]) -> float:
    """Tính Intersection over Union (IoU) giữa 2 span ký tự (start, end)."""
    start_a, end_a = span_a
    start_b, end_b = span_b

    inter_start = max(start_a, start_b)
    inter_end = min(end_a, end_b)

    if inter_start >= inter_end:
        return 0.0

    intersection = inter_end - inter_start
    union = (end_a - start_a) + (end_b - start_b) - intersection
    return intersection / union if union > 0 else 0.0

def evaluate_triplet_extraction(
    gold_triplets_list: List[List[Dict[str, Any]]],
    pred_triplets_list: List[List[Dict[str, Any]]],
    aspect_categories: List[str] = None
) -> Dict[str, Any]:
    """
    Tính Precision, Recall, F1 cho cả Exact Match và Overlap Match (IoU >= 0.5),
    đồng thời phân rã F1 theo từng Aspect Category.
    """
    if aspect_categories is None:
        aspect_categories = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]

    # Counters cho Exact Match
    exact_tp = 0
    exact_fp = 0
    exact_fn = 0

    # Counters cho Overlap Match (IoU >= 0.5)
    overlap_tp = 0
    overlap_fp = 0
    overlap_fn = 0

    # Per-aspect counters
    aspect_stats = {asp: {"tp_exact": 0, "fp_exact": 0, "fn_exact": 0, "support": 0} for asp in aspect_categories}

    for golds, preds in zip(gold_triplets_list, pred_triplets_list):
        # 1. Exact Match
        # Biểu diễn triplet dạng tuple: (aspect, sentiment, start, end)
        gold_tuples = set((g["aspect"], g["sentiment"], g["start"], g["end"]) for g in golds)
        pred_tuples = set((p["aspect"], p["sentiment"], p["start"], p["end"]) for p in preds)

        tp_e = len(gold_tuples.intersection(pred_tuples))
        fp_e = len(pred_tuples - gold_tuples)
        fn_e = len(gold_tuples - pred_tuples)

        exact_tp += tp_e
        exact_fp += fp_e
        exact_fn += fn_e

        # Per-aspect tracking
        for g in gold_tuples:
            asp = g[0]
            if asp in aspect_stats:
                aspect_stats[asp]["support"] += 1
                if g in pred_tuples:
                    aspect_stats[asp]["tp_exact"] += 1
                else:
                    aspect_stats[asp]["fn_exact"] += 1

        for p in pred_tuples:
            asp = p[0]
            if asp in aspect_stats and p not in gold_tuples:
                aspect_stats[asp]["fp_exact"] += 1

        # 2. Overlap Match (IoU >= 0.5)
        matched_gold_indices = set()
        for p in preds:
            matched = False
            for g_idx, g in enumerate(golds):
                if g_idx in matched_gold_indices:
                    continue
                if p["aspect"] == g["aspect"] and p["sentiment"] == g["sentiment"]:
                    iou = compute_span_iou((p["start"], p["end"]), (g["start"], g["end"]))
                    if iou >= 0.5:
                        overlap_tp += 1
                        matched_gold_indices.add(g_idx)
                        matched = True
                        break
            if not matched:
                overlap_fp += 1

        overlap_fn += len(golds) - len(matched_gold_indices)

    def calc_prf(tp: int, fp: int, fn: int) -> Tuple[float, float, float]:
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        return prec, rec, f1

    exact_p, exact_r, exact_f1 = calc_prf(exact_tp, exact_fp, exact_fn)
    overlap_p, overlap_r, overlap_f1 = calc_prf(overlap_tp, overlap_fp, overlap_fn)

    per_aspect_f1 = {}
    for asp, cnt in aspect_stats.items():
        _, _, asp_f1 = calc_prf(cnt["tp_exact"], cnt["fp_exact"], cnt["fn_exact"])
        per_aspect_f1[asp] = {
            "F1": round(asp_f1 * 100, 2),
            "Support": cnt["support"]
        }

    return {
        "exact_match": {
            "precision": round(exact_p * 100, 2),
            "recall": round(exact_r * 100, 2),
            "f1": round(exact_f1 * 100, 2)
        },
        "overlap_match": {
            "precision": round(overlap_p * 100, 2),
            "recall": round(overlap_r * 100, 2),
            "f1": round(overlap_f1 * 100, 2)
        },
        "per_aspect": per_aspect_f1
    }

def evaluate_rating_prediction(
    gold_ratings: List[float],
    pred_ratings: List[float]
) -> Dict[str, float]:
    """Tính các metric định lượng cho Rating: MAE, RMSE, Pearson r, Within-1."""
    golds = np.array(gold_ratings)
    preds = np.array(pred_ratings)

    mae = np.mean(np.abs(golds - preds))
    rmse = np.sqrt(np.mean((golds - preds) ** 2))

    # Pearson correlation
    if np.std(golds) > 1e-6 and np.std(preds) > 1e-6:
        pearson_r = np.corrcoef(golds, preds)[0, 1]
    else:
        pearson_r = 0.0

    # Within-1 accuracy (|y - y_hat| <= 1.0)
    within_1 = np.mean(np.abs(golds - preds) <= 1.0) * 100

    return {
        "mae": round(float(mae), 4),
        "rmse": round(float(rmse), 4),
        "pearson_r": round(float(pearson_r), 4),
        "within_1_acc": round(float(within_1), 2)
    }

def print_evaluation_summary(
    model_name: str,
    overall_results: Dict[str, Any],
    conflict_results: Dict[str, Any] = None,
    consistent_results: Dict[str, Any] = None
) -> None:
    """In bảng tổng kết kết quả đánh giá đẹp mắt, rõ ràng."""
    print("\n" + "=" * 70)
    print(f" BẢNG ĐÁNH GIÁ HIỆU NĂNG MÔ HÌNH: {model_name}")
    print("=" * 70)

    em = overall_results["extraction"]["exact_match"]
    om = overall_results["extraction"]["overlap_match"]
    rt = overall_results["rating"]

    print(" 1. KẾT QUẢ TRÍCH XUẤT Ý KIẾN (OPINION EXTRACTION - ASTE):")
    print(f"    - Exact Triplet Match   : P = {em['precision']:>5.2f}% | R = {em['recall']:>5.2f}% | F1 = {em['f1']:>5.2f}%")
    print(f"    - Overlap Triplet Match : P = {om['precision']:>5.2f}% | R = {om['recall']:>5.2f}% | F1 = {om['f1']:>5.2f}%")
    print("    - F1 theo từng khía cạnh (Per-Aspect Exact F1):")
    for asp, val in overall_results["extraction"]["per_aspect"].items():
        print(f"       * {asp:12s}: F1 = {val['F1']:>5.2f}% (Support: {val['Support']:>4,})")

    print("\n 2. KẾT QUẢ DỰ ĐOÁN RATING TỔNG THỂ (OVERALL RATING PREDICTION):")
    print(f"    - MAE             : {rt['mae']:.4f}")
    print(f"    - RMSE            : {rt['rmse']:.4f}")
    print(f"    - Pearson r       : {rt['pearson_r']:.4f}")
    print(f"    - Within-1 Acc    : {rt['within_1_acc']:.2f}%")

    if conflict_results and consistent_results:
        print("\n 3. PHÂN TÍCH CHẨN ĐOÁN XUNG ĐỘT (CONFLICT SUBSET DIAGNOSTIC):")
        c_f1 = conflict_results["extraction"]["exact_match"]["f1"]
        c_mae = conflict_results["rating"]["mae"]
        s_f1 = consistent_results["extraction"]["exact_match"]["f1"]
        s_mae = consistent_results["rating"]["mae"]

        print(f"    - Consistent Subset : Exact F1 = {s_f1:>5.2f}% | Rating MAE = {s_mae:.4f}")
        print(f"    - Conflict Subset   : Exact F1 = {c_f1:>5.2f}% | Rating MAE = {c_mae:.4f}")
        print(f"    - Delta (Conflict - Consistent): Delta F1 = {c_f1 - s_f1:>+5.2f}% | Delta MAE = {c_mae - s_mae:>+5.4f}")

    print("=" * 70 + "\n")


### 7.4. Quá trình Huấn luyện (Training Loop)

Khởi tạo vòng lặp huấn luyện, tối ưu hóa trọng số mô hình bằng AdamW.


In [ ]:
# =====================================================================
# 1. Dataset & Token Alignment
# =====================================================================
class HotelABSA_Dataset(Dataset):
    """Dataset đọc dữ liệu Hotel-Disjoint JSONL và align spans sang token labels."""
    def __init__(self, table_name: str, tokenizer, max_length: int = 512, db_path="Data/db/chatbot.db"):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        
        import sqlite3
        import pandas as pd
        import json
        conn = sqlite3.connect(db_path)
        df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
        for _, row in df.iterrows():
            item = row.to_dict()
            if isinstance(item.get("triplets"), str):
                item["triplets"] = json.loads(item["triplets"])
            self.samples.append(item)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        text = item["text"]
        triplets = item["triplets"]
        rating = float(item["rating"])

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)
        offset_mapping = encoding["offset_mapping"].squeeze(0)

        # Gán nhãn BIO cho từng token
        seq_len = len(input_ids)
        labels = [LABEL2ID["O"]] * seq_len

        for token_idx, (start_char, end_char) in enumerate(offset_mapping):
            # Special tokens ([CLS], [SEP], PAD)
            if start_char == 0 and end_char == 0:
                labels[token_idx] = -100
                continue

            # Tìm xem token này có thuộc span nào không
            for trip in triplets:
                t_start = trip["start"]
                t_end = trip["end"]
                asp = trip["aspect"]
                sent = trip["sentiment"]

                # Kiểm tra token giao với triplet span
                if start_char >= t_start and end_char <= t_end:
                    # Token đầu tiên của span gán B, các token tiếp theo gán I
                    if start_char == t_start or token_idx == 0 or labels[token_idx - 1] == LABEL2ID["O"] or labels[token_idx - 1] == -100:
                        lbl_name = f"B-{asp}-{sent}"
                    else:
                        lbl_name = f"I-{asp}-{sent}"

                    if lbl_name in LABEL2ID:
                        labels[token_idx] = LABEL2ID[lbl_name]
                    break

        item_dict = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "extraction_labels": torch.tensor(labels, dtype=torch.long),
            "rating_labels": torch.tensor(rating, dtype=torch.float),
            "raw_text": text,
            "gold_triplets": triplets,
            "offset_mapping": offset_mapping
        }

        if "token_type_ids" in encoding:
            item_dict["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        return item_dict

def collate_fn(batch):
    """Custom collate function để xử lý metadata và tensors."""
    input_ids = torch.stack([b["input_ids"] for b in batch])
    attention_mask = torch.stack([b["attention_mask"] for b in batch])
    extraction_labels = torch.stack([b["extraction_labels"] for b in batch])
    rating_labels = torch.stack([b["rating_labels"] for b in batch])

    res = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "extraction_labels": extraction_labels,
        "rating_labels": rating_labels,
        "raw_text": [b["raw_text"] for b in batch],
        "gold_triplets": [b["gold_triplets"] for b in batch],
        "offset_mapping": [b["offset_mapping"] for b in batch]
    }

    if "token_type_ids" in batch[0]:
        res["token_type_ids"] = torch.stack([b["token_type_ids"] for b in batch])

    return res

# =====================================================================
# 2. Decode BIO Predictions To Character Spans
# =====================================================================
def decode_bio_to_triplets(
    pred_logits: torch.Tensor,
    offset_mapping: torch.Tensor,
    raw_text: str
) -> List[Dict[str, Any]]:
    """Giải mã chuỗi nhãn BIO thành danh sách bộ ba (Aspect, Sentiment, Span)."""
    pred_ids = torch.argmax(pred_logits, dim=-1).cpu().numpy()
    offsets = offset_mapping.cpu().numpy()

    extracted = []
    current_span = None

    for idx, (lbl_id, (start_char, end_char)) in enumerate(zip(pred_ids, offsets)):
        if start_char == 0 and end_char == 0:
            continue

        lbl_name = ID2LABEL.get(int(lbl_id), "O")

        if lbl_name.startswith("B-"):
            if current_span is not None:
                extracted.append(current_span)

            parts = lbl_name[2:].split("-")
            asp = parts[0]
            sent = parts[1] if len(parts) > 1 else "Positive"

            current_span = {
                "aspect": asp,
                "sentiment": sent,
                "start": int(start_char),
                "end": int(end_char),
                "span_text": raw_text[start_char:end_char]
            }
        elif lbl_name.startswith("I-") and current_span is not None:
            parts = lbl_name[2:].split("-")
            asp = parts[0]
            if asp == current_span["aspect"]:
                current_span["end"] = int(end_char)
                current_span["span_text"] = raw_text[current_span["start"]:end_char]
            else:
                extracted.append(current_span)
                current_span = None
        else:
            if current_span is not None:
                extracted.append(current_span)
                current_span = None

    if current_span is not None:
        extracted.append(current_span)

    return extracted

# =====================================================================
# 3. Validation Loop
# =====================================================================
def evaluate_model(model, dataloader, device):
    model.eval()
    all_gold_triplets = []
    all_pred_triplets = []
    all_gold_ratings = []
    all_pred_ratings = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            ext_labels = batch["extraction_labels"].to(device)
            rate_labels = batch["rating_labels"].to(device)

            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                extraction_labels=ext_labels,
                rating_labels=rate_labels
            )

            total_loss += outputs["loss"].item()

            ext_logits = outputs["extraction_logits"]
            rate_preds = outputs["rating_preds"]

            # Decode triplets
            for i in range(len(batch["raw_text"])):
                text = batch["raw_text"][i]
                golds = batch["gold_triplets"][i]
                offsets = batch["offset_mapping"][i]
                logits = ext_logits[i]

                preds = decode_bio_to_triplets(logits, offsets, text)
                all_pred_triplets.append(preds)
                all_gold_triplets.append(golds)

            all_gold_ratings.extend(rate_labels.cpu().numpy().tolist())
            all_pred_ratings.extend(rate_preds.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0.0
    ext_results = evaluate_triplet_extraction(all_gold_triplets, all_pred_triplets)
    rate_results = evaluate_rating_prediction(all_gold_ratings, all_pred_ratings)

    return {
        "loss": avg_loss,
        "extraction": ext_results,
        "rating": rate_results
    }

# =====================================================================
# 4. Main Training Function
# =====================================================================
def train_model(
    model_type: str = "cram",
    model_name_or_path: str = "microsoft/deberta-v3-base",
    data_dir: str = "Data/split",
    output_dir: str = "checkpoints",
    epochs: int = 5,
    batch_size: int = 8,
    lr: float = 2e-5,
    max_length: int = 256,
    lambda_ext: float = 1.0,
    lambda_rate: float = 0.5,
    seed: int = 42
):
    """Huấn luyện CRAM_ABSA hoặc PlainMTL_ABSA."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"[DEVICE] Sử dụng NVIDIA GPU: {torch.cuda.get_device_name(0)}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("[DEVICE] Sử dụng Apple Silicon (MPS)")
    else:
        device = torch.device("cpu")
        print("[DEVICE] Sử dụng CPU")

    os.makedirs(output_dir, exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

    print("\n--- Nạp Datasets ---")
    train_dataset = HotelABSA_Dataset("train_split", tokenizer, max_length)
    val_dataset = HotelABSA_Dataset("val_split", tokenizer, max_length)
    test_dataset = HotelABSA_Dataset("test_split", tokenizer, max_length)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

    # Khởi tạo mô hình
    if model_type.lower() == "cram":
        print("\n--- Khởi tạo Proposed Model: CRAM_ABSA ---")
        model = CRAM_ABSA(model_name_or_path=model_name_or_path)
    else:
        print("\n--- Khởi tạo Baseline Model: PlainMTL_ABSA ---")
        model = PlainMTL_ABSA(model_name_or_path=model_name_or_path)

    model.to(device)

    # Optimizer & Scheduler
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": 0.01,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]

    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=lr)
    total_steps = len(train_loader) * epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    best_score = -float("inf")
    best_checkpoint_path = os.path.join(output_dir, f"best_{model_type}.pt")

    print("\n" + "=" * 60)
    print(f" BẮT ĐẦU HUẤN LUYỆN {model_type.upper()} ({epochs} EPOCHS)")
    print("=" * 60)

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0

        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            ext_labels = batch["extraction_labels"].to(device)
            rate_labels = batch["rating_labels"].to(device)

            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                extraction_labels=ext_labels,
                rating_labels=rate_labels,
                lambda_ext=lambda_ext,
                lambda_rate=lambda_rate
            )

            loss = outputs["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

            if (step + 1) % 100 == 0 or (step + 1) == len(train_loader):
                print(f"Epoch [{epoch}/{epochs}] | Step [{step+1}/{len(train_loader)}] | Train Loss: {loss.item():.4f}")

        avg_train_loss = train_loss / len(train_loader)
        val_res = evaluate_model(model, val_loader, device)

        val_exact_f1 = val_res["extraction"]["exact_match"]["f1"]
        val_overlap_f1 = val_res["extraction"]["overlap_match"]["f1"]
        val_mae = val_res["rating"]["mae"]

        # Composite score để chọn checkpoint tốt nhất
        composite_score = 0.5 * val_overlap_f1 + 0.5 * (1.0 - val_mae / 5.0) * 100
        epoch_time = time.time() - t0

        print(f"\n>> Epoch {epoch} Hoàn tất ({epoch_time:.1f}s):")
        print(f"   Train Loss : {avg_train_loss:.4f} | Val Loss: {val_res['loss']:.4f}")
        print(f"   Val Overlap F1: {val_overlap_f1:.2f}% | Exact F1: {val_exact_f1:.2f}% | Rating MAE: {val_mae:.4f}")
        print(f"   Composite Score: {composite_score:.2f}")

        if composite_score > best_score:
            best_score = composite_score
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "composite_score": composite_score,
                "val_results": val_res,
                "model_type": model_type,
                "config": model.config
            }, best_checkpoint_path)
            print(f"   [SAVED] Đã lưu checkpoint mới tốt nhất: {best_checkpoint_path}\n")

    print("\n" + "=" * 60)
    print(" HUẤN LUYỆN HOÀN TẤT. BẮT ĐẦU ĐÁNH GIÁ TRÊN OFFICIAL TEST SET...")
    print("=" * 60)

    # Đánh giá checkpoint tốt nhất trên Test Set
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    test_res = evaluate_model(model, test_loader, device)

    # Đánh giá riêng Conflict Subset vs Consistent Subset
    conflict_dataset = HotelABSA_Dataset("test_conflict_split", tokenizer, max_length)
    consistent_dataset = HotelABSA_Dataset("test_consistent_split", tokenizer, max_length)

    conflict_loader = DataLoader(conflict_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    consistent_loader = DataLoader(consistent_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    conflict_res = evaluate_model(model, conflict_loader, device)
    consistent_res = evaluate_model(model, consistent_loader, device)

    print_evaluation_summary(
        model_name=f"{model_type.upper()} ({model_name_or_path})",
        overall_results=test_res,
        conflict_results=conflict_res,
        consistent_results=consistent_res
    )

    return test_res

if __name__ == "__main__":
    # Test chạy thử pipeline
    print("Train module sẵn sàng!")



## 9. Huấn luyện mô hình và Tải về (Dành riêng cho Google Colab)
Chạy cell dưới đây để bắt đầu huấn luyện mô hình CRAM-ABSA và tự động tải file weights (`best_cram.pt`) về máy sau khi hoàn tất.

In [ ]:
# 1. Bắt đầu quá trình huấn luyện tất cả các mô hình
models_to_train = ["plainmtl", "cram"]
all_results = {}

for m_type in models_to_train:
    print(f"\n{=*50}")
    print(f" BẮT ĐẦU TRAINING MODEL {m_type.upper()} TRÊN COLAB")
    print(f"{=*50}\n")
    
    res = train_model(
        model_type=m_type,
        model_name_or_path="microsoft/deberta-v3-base",
        data_dir="Data/split",
        output_dir="checkpoints",
        epochs=5,           # Bạn có thể tăng số epoch nếu muốn
        batch_size=8,       # Điều chỉnh tuỳ theo GPU (Colab T4 thường chịu được batch_size=8 hoặc 16)
        lr=2e-5
    )
    all_results[m_type] = res

print("\n=== KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP TRÊN TẬP TEST ===")
for m_type, res in all_results.items():
    print(f"\n--- MÔ HÌNH: {m_type.upper()} ---")
    print(res)


In [ ]:
# 2. Tự động tải tất cả file model về máy cá nhân
try:
    from google.colab import files
    import os
    import time
    
    for m_type in models_to_train:
        model_path = f"checkpoints/best_{m_type}.pt"
        if os.path.exists(model_path):
            print(f"Đang tải file {model_path} về máy...")
            files.download(model_path)
            time.sleep(2)  # Đợi một chút giữa các lần tải
        else:
            print(f"Không tìm thấy file {model_path}. Quá trình train có thể đã gặp lỗi.")
    print("\n(Vui lòng không đóng tab Colab cho đến khi quá trình tải tất cả các file hoàn tất)")
except ImportError:
    print("Lỗi: Đoạn code này chỉ hoạt động trên môi trường Google Colab!")
